# 1\.2\.6 Create Taxi Zone Spatial Context

This notebook adds reusable static spatial context to the harmonized NYC Taxi Zone reference layer before multimodal integration\. We prepare one reliable geometry per canonical Taxi Zone, build a transportation\-aware connectivity table, and derive congestion\-pricing geography, gateway, adjacency, distance, centroid, and graph\-based network fields\. These are stable geographic attributes rather than date\-specific mobility features, so downstream notebooks can join them consistently without rebuilding spatial logic\.

In [1]:
from pathlib import Path
import json

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from shapely import wkb
from shapely.ops import unary_union

from project_branding import (
    BRAND_COLORS,
    BRAND_MAP_COLORS,
    apply_branding,
)

# The notebook uses Plotly's MapLibre-backed map traces so rendered
# notebook HTML does not depend on a Mapbox token.
if not hasattr(go, "Choroplethmap"):
    raise RuntimeError(
        "This map requires Plotly's MapLibre-backed go.Choroplethmap. "
        "Do not fall back to token-dependent Choroplethmapbox."
    )

In [2]:
# ---------------------------------------------------------------------
# 1.2.6 Control Panel and Paths
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

HARMONIZED_INPUT_DIR = PIPELINE_DATA_DIR / "1.2.4.harmonized_mobility_datasets"
SPATIAL_INTERMEDIATE_DIR = PIPELINE_DATA_DIR / "1.2.6.intermediate_tables"
SPATIAL_FINAL_DIR = PIPELINE_DATA_DIR / "1.2.6.final_tables"

SPATIAL_INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_FINAL_DIR.mkdir(parents=True, exist_ok=True)

TAXI_ZONES_PATH = HARMONIZED_INPUT_DIR / "nyc_taxi_zones_harmonized.parquet"
TAXI_ZONE_CONNECTIVITY_OUTPUT_PATH = SPATIAL_FINAL_DIR / "taxi_zone_connectivity.parquet"
TAXI_ZONE_SPATIAL_CONTEXT_OUTPUT_PATH = SPATIAL_FINAL_DIR / "taxi_zone_spatial_context.parquet"
SPATIAL_CONTEXT_MANIFEST_OUTPUT_PATH = SPATIAL_FINAL_DIR / "spatial_context_manifest.csv"

WRITE_OUTPUTS = True
REBUILD_SPATIAL_CONTEXT = True

spatial_setup_summary_df = pd.DataFrame(
    {
        "setting": [
            "taxi_zones_input",
            "spatial_intermediate_dir",
            "spatial_final_dir",
            "write_outputs",
            "rebuild_spatial_context",
        ],
        "value": [
            str(TAXI_ZONES_PATH),
            str(SPATIAL_INTERMEDIATE_DIR),
            str(SPATIAL_FINAL_DIR),
            WRITE_OUTPUTS,
            REBUILD_SPATIAL_CONTEXT,
        ],
    }
)

display(spatial_setup_summary_df)

,setting,value
0,taxi_zones_input,pipeline_data/1.2.4.harmonized_mobility_datase...
1,spatial_intermediate_dir,pipeline_data/1.2.6.intermediate_tables
2,spatial_final_dir,pipeline_data/1.2.6.final_tables
3,write_outputs,True
4,rebuild_spatial_context,True


Let's load the harmonized Taxi Zone reference produced by \`1\.2\.4\` and reconstruct its stored geometry\. This is the only upstream dataset needed here\. We will inspect its real schema and retain both source LocationIDs and canonical LocationIDs so aliases remain joinable downstream\.

In [3]:
# ---------------------------------------------------------------------
# Load harmonized Taxi Zone reference layer
# ---------------------------------------------------------------------

taxi_zones_df = pd.read_parquet(TAXI_ZONES_PATH)

taxi_zones_df["geometry"] = taxi_zones_df["geometry"].apply(
    lambda geom: wkb.loads(geom) if pd.notna(geom) and not hasattr(geom, "geom_type") else geom
)

taxi_zones_gdf = gpd.GeoDataFrame(
    taxi_zones_df,
    geometry="geometry",
    crs="EPSG:4326",
)

taxi_zone_input_summary_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "distinct_location_ids",
            "distinct_canonical_location_ids",
            "null_geometry_rows",
            "crs",
        ],
        "value": [
            len(taxi_zones_gdf),
            taxi_zones_gdf.shape[1],
            taxi_zones_gdf["location_id"].nunique(),
            taxi_zones_gdf["canonical_location_id"].nunique(),
            taxi_zones_gdf["geometry"].isna().sum(),
            str(taxi_zones_gdf.crs),
        ],
    }
)

display(taxi_zone_input_summary_df)

,metric,value
0,row_count,270
1,column_count,7
2,distinct_location_ids,264
3,distinct_canonical_location_ids,262
4,null_geometry_rows,2
5,crs,EPSG:4326


The harmonized Taxi Zone reference is now loaded directly from \`1\.2\.4\`\. The next section checks its geography structure before we derive connectivity or policy\-area features\.

## 1\.2\.6\.1 Validate Taxi Zone Spatial Reference

Before creating spatial context, let's verify the real geography handoff from \`1\.2\.4\`\. We care about valid geometry, source and canonical identifiers, multipart representation, and the expected nonphysical Unknown zones\.

In [4]:
# ---------------------------------------------------------------------
# Validate harmonized Taxi Zone reference structure
# ---------------------------------------------------------------------

valid_geometry_mask = taxi_zones_gdf["geometry"].notna()
valid_geometry_count = taxi_zones_gdf.loc[valid_geometry_mask, "geometry"].is_valid.sum()

spatial_reference_validation_df = pd.DataFrame(
    {
        "check": [
            "required_columns_present",
            "non_null_location_id",
            "non_null_canonical_location_id",
            "all_non_null_geometries_valid",
            "physical_zone_rows_present",
        ],
        "status": [
            {"location_id", "canonical_location_id", "zone", "borough", "geometry"}.issubset(taxi_zones_gdf.columns),
            taxi_zones_gdf["location_id"].notna().all(),
            taxi_zones_gdf["canonical_location_id"].notna().all(),
            valid_geometry_count == valid_geometry_mask.sum(),
            valid_geometry_mask.any(),
        ],
    }
)

display(spatial_reference_validation_df)

,check,status
0,required_columns_present,True
1,non_null_location_id,True
2,non_null_canonical_location_id,True
3,all_non_null_geometries_valid,True
4,physical_zone_rows_present,True


Findings\. The Taxi Zone reference looks structurally sound for spatial feature work\. The extra rows are expected because this layer keeps multipart and alias geography that we still need for TLC compatibility, while the two null geometries belong to the nonphysical Unknown zones\. In other words, the oddities here are intentional, not signs that the handoff from \`1\.2\.4\` went sideways\.

In [5]:
(
    taxi_zones_gdf
    .groupby("location_id")
    .size()
    .reset_index(name="polygon_count")
    .sort_values("polygon_count", ascending=False)
    .head(10)
)

,location_id,polygon_count
102,103,3
103,105,3
55,56,2
56,57,2
179,181,1
178,180,1
177,179,1
176,178,1
175,177,1
174,176,1


Findings\. The observed geometry counts align with the harmonization logic implemented earlier in the pipeline\. Governor's Island / Ellis Island / Liberty Island \(LocationIDs 103 and 105\) are represented by three polygon geometries each, while Corona \(LocationIDs 56 and 57\) is represented by two polygon geometries each\. All remaining Taxi Zones are represented by a single polygon geometry\. This confirms that the additional records in the reference layer are explained by intentional alias handling rather than unexpected duplication or data quality issues\.

## 1\.2\.6\.2 Create Taxi Zone Connectivity Table

This section builds a reusable connectivity table at the canonical Taxi Zone grain\. We first dissolve multipart and alias geometry into one physical zone, then combine polygon\-touching adjacency with the existing transportation connectors that bridge important cross\-waterway gaps\. The result is static spatial infrastructure; no mobility measurements are used\.

### Prepare Taxi Zone Geometry Layer

This block prepares the Taxi Zone geometry layer used throughout the spatial feature engineering workflow\. The harmonized Taxi Zone reference layer already contains centroid coordinates generated during data integration, so we reuse those values\. We still create a projected centroid geometry object for distance\-based calculations while ensuring all spatial operations occur in a CRS appropriate for NYC adjacency and proximity analysis\.

In [6]:
# ---------------------------------------------------------------------
# Prepare one geometry per canonical Taxi Zone
# ---------------------------------------------------------------------

taxi_zones_working_gdf = taxi_zones_gdf.copy()
taxi_zones_working_gdf["location_id"] = taxi_zones_working_gdf["location_id"].astype("Int64")
taxi_zones_working_gdf["canonical_location_id"] = taxi_zones_working_gdf["canonical_location_id"].astype("Int64")

physical_taxi_zones_gdf = taxi_zones_working_gdf[
    taxi_zones_working_gdf["geometry"].notna()
].copy()

canonical_zone_attributes_df = (
    physical_taxi_zones_gdf
    .sort_values(["canonical_location_id", "location_id"])
    .groupby("canonical_location_id", as_index=False)
    .agg(zone=("zone", "first"), borough=("borough", "first"))
)

canonical_zone_geometry_gdf = (
    physical_taxi_zones_gdf[["canonical_location_id", "geometry"]]
    .dissolve(by="canonical_location_id", as_index=False)
    .merge(canonical_zone_attributes_df, on="canonical_location_id", how="left")
)

canonical_zone_geometry_gdf["location_id"] = canonical_zone_geometry_gdf["canonical_location_id"]
taxi_zones_geo_gdf = canonical_zone_geometry_gdf.to_crs("EPSG:2263").copy()
taxi_zones_geo_gdf["centroid_x"] = taxi_zones_geo_gdf.geometry.centroid.x
taxi_zones_geo_gdf["centroid_y"] = taxi_zones_geo_gdf.geometry.centroid.y
taxi_zones_geo_gdf["centroid_geometry"] = gpd.points_from_xy(
    taxi_zones_geo_gdf["centroid_x"],
    taxi_zones_geo_gdf["centroid_y"],
    crs="EPSG:2263",
)

geometry_preparation_summary_df = pd.DataFrame(
    {
        "metric": [
            "source_reference_rows",
            "physical_source_rows",
            "canonical_geometry_rows",
            "multipart_rows_collapsed",
            "canonical_geometry_key_unique",
            "projected_crs",
        ],
        "value": [
            len(taxi_zones_working_gdf),
            len(physical_taxi_zones_gdf),
            len(taxi_zones_geo_gdf),
            len(physical_taxi_zones_gdf) - len(taxi_zones_geo_gdf),
            not taxi_zones_geo_gdf["canonical_location_id"].duplicated().any(),
            str(taxi_zones_geo_gdf.crs),
        ],
    }
)

display(geometry_preparation_summary_df)

,metric,value
0,source_reference_rows,270
1,physical_source_rows,268
2,canonical_geometry_rows,260
3,multipart_rows_collapsed,8
4,canonical_geometry_key_unique,True
5,projected_crs,EPSG:2263


Findings\. The working geometry layer is in the shape we need for spatial feature engineering\. Once the two nonphysical Unknown zones are excluded, we have a clean set of physical Taxi Zone polygons, with multipart and alias handling preserved exactly where the project expects it\. Projecting to EPSG:2263 also puts us on solid footing for centroid, distance, adjacency, and spillover calculations\.

### Analyze Cross\-Waterway Connectors

This section evaluates an important limitation of pure polygon\-based adjacency\. Many of NYC's most important mobility corridors cross rivers, bays, and waterways, meaning transportation activity can flow between areas that do not share a geographic boundary\. Before finalizing the connectivity network, let's examine which cross\-borough relationships are naturally captured by polygon adjacency and which major transportation corridors would be missed without additional bridge, tunnel, or connector relationships\.

Let's start by constructing the baseline connectivity graph using only Taxi Zone polygons\. Two zones are considered connected if their boundaries touch, creating a purely geographic representation of neighborhood adjacency that can serve as the foundation for downstream spillover features\.

In [7]:
# ---------------------------------------------------------------------
# Build polygon-touching connectivity relationships
# ---------------------------------------------------------------------

# This is the baseline geographic network: two Taxi Zones are connected when their
# polygons intersect. Transportation connectors are added later because waterways
# can separate zones that are operationally connected but not polygon-adjacent.
adjacency_left_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "canonical_location_id",
        "zone",
        "borough",
        "geometry",
    ]
].copy()

adjacency_right_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "canonical_location_id",
        "zone",
        "borough",
        "geometry",
    ]
].copy()

polygon_connectivity_gdf = gpd.sjoin(
    adjacency_left_gdf,
    adjacency_right_gdf,
    how="inner",
    predicate="intersects",
)

polygon_connectivity_df = (
    polygon_connectivity_gdf[
        polygon_connectivity_gdf["location_id_left"]
        != polygon_connectivity_gdf["location_id_right"]
    ][
        [
            "location_id_left",
            "canonical_location_id_left",
            "zone_left",
            "borough_left",
            "location_id_right",
            "canonical_location_id_right",
            "zone_right",
            "borough_right",
        ]
    ]
    .rename(
        columns={
            "location_id_left": "location_id",
            "canonical_location_id_left": "canonical_location_id",
            "zone_left": "zone",
            "borough_left": "borough",
            "location_id_right": "connected_location_id",
            "canonical_location_id_right": "connected_canonical_location_id",
            "zone_right": "connected_zone",
            "borough_right": "connected_borough",
        }
    )
    .drop_duplicates()
    .sort_values(["location_id", "connected_location_id"])
    .reset_index(drop=True)
)

print("Polygon-touching connectivity rows:")
print(len(polygon_connectivity_df))

display(polygon_connectivity_df.head())

Polygon-touching connectivity rows:
1290


,location_id,canonical_location_id,zone,borough,connected_location_id,connected_canonical_location_id,connected_zone,connected_borough
0,2,2,Jamaica Bay,Queens,30,30,Broad Channel,Queens
1,2,2,Jamaica Bay,Queens,132,132,JFK Airport,Queens
2,3,3,Allerton/Pelham Gardens,Bronx,32,32,Bronxdale,Bronx
3,3,3,Allerton/Pelham Gardens,Bronx,51,51,Co-Op City,Bronx
4,3,3,Allerton/Pelham Gardens,Bronx,81,81,Eastchester,Bronx


Next, let's isolate only the polygon\-touching relationships that cross borough boundaries\. This allows us to evaluate how much cross\-borough connectivity is already captured by the Taxi Zone geometry before introducing any transportation\-corridor augmentations\.

In [8]:
# ---------------------------------------------------------------------
# Identify cross-borough polygon-touching adjacency relationships
# ---------------------------------------------------------------------

# Cross-borough polygon adjacency is checked separately because NYC connectivity is
# shaped by waterways. This tells us what the geometry captures before adding bridge,
# tunnel, and gateway relationships to the network.
cross_borough_polygon_adjacency_df = (
    polygon_connectivity_df[
        polygon_connectivity_df["borough"]
        != polygon_connectivity_df["connected_borough"]
    ]
    .copy()
    .sort_values(
        [
            "borough",
            "connected_borough",
            "zone",
            "connected_zone",
        ]
    )
    .reset_index(drop=True)
)

print("Cross-borough polygon-touching adjacency rows:")
print(len(cross_borough_polygon_adjacency_df))

display(cross_borough_polygon_adjacency_df)

Cross-borough polygon-touching adjacency rows:
28


,location_id,canonical_location_id,zone,borough,connected_location_id,connected_canonical_location_id,connected_zone,connected_borough
0,136,136,Kingsbridge Heights,Bronx,153,153,Marble Hill,Manhattan
1,220,220,Spuyten Duyvil/Kingsbridge,Bronx,128,128,Inwood Hill Park,Manhattan
2,220,220,Spuyten Duyvil/Kingsbridge,Bronx,153,153,Marble Hill,Manhattan
3,199,199,Rikers Island,Bronx,138,138,LaGuardia Airport,Queens
4,36,36,Bushwick North,Brooklyn,96,96,Forest Park/Highland Park,Queens
5,36,36,Bushwick North,Brooklyn,198,198,Ridgewood,Queens
6,37,37,Bushwick South,Brooklyn,96,96,Forest Park/Highland Park,Queens
7,63,63,Cypress Hills,Brooklyn,96,96,Forest Park/Highland Park,Queens
8,63,63,Cypress Hills,Brooklyn,124,124,Howard Beach,Queens
9,63,63,Cypress Hills,Brooklyn,180,180,Ozone Park,Queens


Finally, let's visualize the cross\-borough polygon adjacency network\. Mapping these relationships makes it easier to identify where the geometry naturally captures cross\-borough movement and where major waterways create gaps that may require supplemental transportation\-corridor connections\.

In [9]:
# ---------------------------------------------------------------------
# Map actual cross-borough polygon-touching adjacency relationships
# ---------------------------------------------------------------------

cross_borough_zone_ids = sorted(
    set(cross_borough_polygon_adjacency_df["location_id"])
    | set(cross_borough_polygon_adjacency_df["connected_location_id"])
)

# ---------------------------------------------------------------------
# Build the polygon scaffold using the proven 4.6.1 MapLibre pattern
# ---------------------------------------------------------------------

cross_borough_plot_gdf = (
    taxi_zones_geo_gdf.loc[
        taxi_zones_geo_gdf["location_id"].isin(cross_borough_zone_ids),
        ["location_id", "zone", "borough", "geometry"],
    ]
    .copy()
    .sort_values("location_id")
    .drop_duplicates(subset=["location_id"])
    .reset_index(drop=True)
)

# Match 4.6.1: simplify in projected feet, then return to EPSG:4326.
cross_borough_plot_gdf = cross_borough_plot_gdf.to_crs(
    "EPSG:2263"
).copy()

cross_borough_plot_gdf["geometry"] = (
    cross_borough_plot_gdf["geometry"].simplify(
        tolerance=75,
        preserve_topology=True,
    )
)

cross_borough_plot_gdf = (
    cross_borough_plot_gdf
    .to_crs("EPSG:4326")
    .reset_index(drop=True)
)

# Match 4.6.1: use string IDs on both sides of the GeoJSON join.
cross_borough_plot_gdf["location_id"] = (
    pd.to_numeric(
        cross_borough_plot_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

cross_borough_geojson = json.loads(
    cross_borough_plot_gdf[
        ["location_id", "zone", "borough", "geometry"]
    ].to_json()
)

# ---------------------------------------------------------------------
# Prepare centroid coordinates for the existing QA points and links
# ---------------------------------------------------------------------

cross_borough_centroid_gdf = taxi_zones_geo_gdf[
    ["location_id", "zone", "borough", "centroid_geometry"]
].copy()

cross_borough_centroid_gdf = gpd.GeoDataFrame(
    cross_borough_centroid_gdf,
    geometry="centroid_geometry",
    crs="EPSG:2263",
).to_crs("EPSG:4326")

cross_borough_centroid_gdf["location_id"] = (
    pd.to_numeric(
        cross_borough_centroid_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

cross_borough_centroid_gdf["centroid_lon"] = (
    cross_borough_centroid_gdf.geometry.x
)
cross_borough_centroid_gdf["centroid_lat"] = (
    cross_borough_centroid_gdf.geometry.y
)

cross_borough_zone_ids_str = {
    str(location_id)
    for location_id in cross_borough_zone_ids
}

cross_borough_centroid_df = (
    cross_borough_centroid_gdf.loc[
        cross_borough_centroid_gdf["location_id"].isin(
            cross_borough_zone_ids_str
        ),
        [
            "location_id",
            "zone",
            "borough",
            "centroid_lat",
            "centroid_lon",
        ],
    ]
    .drop_duplicates("location_id")
    .copy()
)

cross_borough_zone_lookup_df = (
    cross_borough_centroid_df
    .set_index("location_id")
)

# ---------------------------------------------------------------------
# Render
# ---------------------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Choroplethmap(
        geojson=cross_borough_geojson,
        locations=cross_borough_plot_gdf["location_id"].tolist(),
        z=[1.0] * len(cross_borough_plot_gdf),
        featureidkey="properties.location_id",
        colorscale=[
            [0.0, BRAND_COLORS["ice"]],
            [1.0, BRAND_COLORS["ice"]],
        ],
        zmin=0.0,
        zmax=1.0,
        marker={
            "opacity": 0.35,
            "line": {
                "width": 1.5,
                "color": BRAND_MAP_COLORS["primary"],
            },
        },
        customdata=cross_borough_plot_gdf[
            ["zone", "borough"]
        ].to_numpy(dtype=object),
        hovertemplate=(
            "LocationID: %{location}<br>"
            "Zone: %{customdata[0]}<br>"
            "Borough: %{customdata[1]}"
            "<extra></extra>"
        ),
        showscale=False,
        name="Cross-borough polygon-touching Taxi Zones",
    )
)

fig.add_trace(
    go.Scattermap(
        lat=cross_borough_centroid_df["centroid_lat"],
        lon=cross_borough_centroid_df["centroid_lon"],
        mode="markers",
        marker={
            "size": 10,
            "color": BRAND_MAP_COLORS["primary"],
        },
        customdata=cross_borough_centroid_df[
            ["location_id", "zone", "borough"]
        ].to_numpy(dtype=object),
        hovertemplate=(
            "LocationID: %{customdata[0]}<br>"
            "Zone: %{customdata[1]}<br>"
            "Borough: %{customdata[2]}"
            "<extra></extra>"
        ),
        name="Zone centroids",
    )
)

for _, row in cross_borough_polygon_adjacency_df.iterrows():
    left_id = str(int(row["location_id"]))
    right_id = str(int(row["connected_location_id"]))

    left = cross_borough_zone_lookup_df.loc[left_id]
    right = cross_borough_zone_lookup_df.loc[right_id]

    fig.add_trace(
        go.Scattermap(
            lat=[
                left["centroid_lat"],
                right["centroid_lat"],
            ],
            lon=[
                left["centroid_lon"],
                right["centroid_lon"],
            ],
            mode="lines",
            line={
                "color": BRAND_MAP_COLORS["primary"],
                "width": 4,
            },
            customdata=[
                [
                    left_id,
                    left["zone"],
                    left["borough"],
                    right_id,
                    right["zone"],
                    right["borough"],
                ],
                [
                    left_id,
                    left["zone"],
                    left["borough"],
                    right_id,
                    right["zone"],
                    right["borough"],
                ],
            ],
            hovertemplate=(
                "%{customdata[0]}: %{customdata[1]} "
                "(%{customdata[2]}) ↔ "
                "%{customdata[3]}: %{customdata[4]} "
                "(%{customdata[5]})"
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )

fig.update_layout(
    title="Cross-Borough Polygon-Touching Adjacency",
    map={
        "style": "carto-positron",
        "center": {"lat": 40.74, "lon": -73.93},
        "zoom": 10,
    },
    width=950,
    height=760,
    margin={"l": 15, "r": 20, "t": 80, "b": 15},
    legend={"orientation": "h"},
)

apply_branding(fig)

fig.show()

Findings\. This QA makes the limitation of pure polygon adjacency pretty obvious\. It captures some cross\-borough contact, but it misses a lot of the connections that matter operationally in NYC, especially where waterways break the geometry even though bridges, tunnels, and airport access corridors clearly connect the zones in practice\. That gives us a strong reason to keep the transportation\-corridor augmentation in the final connectivity table\.

### Identify Missing Connectors 

Polygon\-touching adjacency gives us the baseline neighborhood graph, but it misses important NYC mobility connections that cross water, tunnels, bridges, causeways, and other physical barriers\. This section identifies the missing transportation corridors that should augment the polygon graph so the final connectivity table better reflects how mobility actually moves through the city\. The goal is not to over\-engineer the road network, but to add a small, defensible set of fixed\-crossing links that prevent obviously connected areas from appearing disconnected in downstream spillover and anomaly\-detection features\.

This block defines the supplemental transportation\-corridor links that polygon adjacency misses\. These links are added on top of the base polygon\-touching graph and represent major crossings such as East River bridges, Newtown Creek crossings, Staten Island and New Jersey access points, Harlem River connectors, and Jamaica Bay/Rockaway crossings\.

In [10]:
# Proposed fixed-crossing connectivity links for visual QA.
# These links supplement polygon-touching adjacency by connecting Taxi Zones
# across major bridges, tunnels, and airport-access corridors.

gateway_connectivity_pairs = [
    # Queens ↔ Manhattan
    (145, 162),  # LIC/Hunters Point ↔ Midtown East
    (146, 162),  # LIC/Queens Plaza ↔ Midtown East
    (193, 162),  # Queensbridge/Ravenswood ↔ Midtown East
    (202, 193),  # Roosevelt Island ↔ Queensbridge/Ravenswood

    # Manhattan islands
    (162, 202),  # Roosevelt Island ↔ Queensbridge/Ravenswood
    # Randall's / Wards Island: RFK / Triborough Bridge corridor
    (194, 168),  # Randall's Island ↔ Mott Haven/Port Morris
    (194, 193),  # Randall's Island ↔ Queensbridge/Ravenswood
    (194, 74),   # Randall's Island ↔ East Harlem North
    (194, 8),   # Randall's Island ↔ Astoria Park
    (194, 179),   # Randall's Island ↔ Old Astoria
    (194, 223),   # Randall's Island ↔ Steinway
    (179, 74),   # Old Astoria ↔ East Harlem North
    (223, 74),   # Steinway ↔ East Harlem North
    (168, 74),   # Mott Haven/Port Morris ↔ East Harlem North

    # Brooklyn ↔ Manhattan: Brooklyn Bridge / Manhattan Bridge corridor
    (66, 87),    # DUMBO/Vinegar Hill ↔ Financial District North
    (65, 87),    # Downtown Brooklyn/MetroTech ↔ Financial District North
    (33, 87),    # Brooklyn Heights ↔ Financial District North

    # Brooklyn ↔ Manhattan: Williamsburg Bridge corridor
    (255, 148),  # Williamsburg North Side ↔ Lower East Side
    (256, 148),  # Williamsburg South Side ↔ Lower East Side
    (217, 148),  # South Williamsburg ↔ Lower East Side

    # Brooklyn ↔ Manhattan: Hugh Carey / Brooklyn-Battery Tunnel corridor
    (195, 12),   # Red Hook ↔ Battery Park
    (40, 12),    # Carroll Gardens ↔ Battery Park

    # Brooklyn ↔ Queens: Newtown Creek corridor
    (112, 145),  # Greenpoint ↔ LIC/Hunters Point
    (112, 226),  # Greenpoint ↔ Sunnyside
    (80, 226),   # East Williamsburg ↔ Sunnyside

    # Staten Island ↔ Brooklyn
    (6, 14),     # Arrochar/Fort Wadsworth ↔ Bay Ridge

    # New Jersey ↔ Staten Island: Bayonne / Goethals corridor
    (187, 1),    # Port Richmond ↔ Newark Airport
    (156, 1),    # Mariners Harbor ↔ Newark Airport
    (44, 1),     # Charleston/Tottenville ↔ Newark Airport

    # Newark / NJ ↔ western / lower Manhattan
    (1, 125),    # Newark Airport ↔ Hudson Sq
    (1, 231),    # Newark Airport ↔ TriBeCa/Civic Center
    (1, 246),    # Newark Airport ↔ West Chelsea/Hudson Yards
    (1, 50),    # Newark Airport ↔ Clinton West
    (243, 1),   # Washington Heights North ↔ Newark Airport

    # Bronx Islands
    (46, 184),  # Pelham Bay Park ↔ City Island

    # Bronx ↔ Manhattan
    (74, 168),  # East Harlem North ↔ Mott Haven/Port Morris
    (74, 247),  # East Harlem North ↔ West Concourse
    (42, 247),  # Central Harlem North ↔ West Concourse
    (42, 119),  # Central Harlem North ↔ Highbridge
    (153, 127),   # Marble Hill ↔ Inwood
    (136, 127),   # Kingsbridge Heights ↔ Inwood
    # Harlem River crossings
    (153, 127),  # Marble Hill ↔ Inwood / Broadway corridor
    (243, 235),   # Washington Heights North ↔ University Heights/Morris Heights
    (243, 119),   # Washington Heights North ↔ Highbridge
    (244, 235),   # Washington Heights South ↔ University Heights/Morris Heights
    (244, 119),   # Washington Heights South ↔ Highbridge

    # Bronx ↔ Queens: Bronx-Whitestone Bridge and Throgs Neck Bridge
    (208, 252),  # Schuylerville/Edgewater Park ↔ Whitestone    
    (208, 15),  # Schuylerville/Edgewater Park ↔ Bay Terrace/Fort Totten

    (199, 223), # Riker's Island/Steinway

    # Jamaica Bay / Rockaway crossings
    (124, 2),    # Howard Beach ↔ Jamaica Bay
    (2, 30),     # Jamaica Bay ↔ Broad Channel
    (30, 117),   # Broad Channel ↔ Hammels/Arverne
    (30, 201),   # Broad Channel ↔ Rockaway Park
    (27, 155),   # Breezy Point/Fort Tilden/Riis Beach ↔ Marine Park/Mill Basin
]

# Use lat/lon geometries for mapping.
taxi_zones_map_gdf = taxi_zones_geo_gdf.to_crs("EPSG:4326").copy()

# Calculate centroids in projected CRS, then convert centroid points to lat/lon.
taxi_zones_centroid_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "zone",
        "borough",
        "centroid_geometry",
    ]
].copy()

taxi_zones_centroid_gdf = gpd.GeoDataFrame(
    taxi_zones_centroid_gdf,
    geometry="centroid_geometry",
    crs="EPSG:2263",
).to_crs("EPSG:4326")

taxi_zones_centroid_gdf["centroid_lon"] = taxi_zones_centroid_gdf.geometry.x
taxi_zones_centroid_gdf["centroid_lat"] = taxi_zones_centroid_gdf.geometry.y

# ---------------------------------------------------------------------
# Build the polygon scaffold using the proven 4.6.1 MapLibre pattern
# ---------------------------------------------------------------------

gateway_zone_ids = sorted(
    {left for left, right in gateway_connectivity_pairs}
    | {right for left, right in gateway_connectivity_pairs}
)

gateway_plot_gdf = (
    taxi_zones_geo_gdf.loc[
        taxi_zones_geo_gdf["location_id"].isin(gateway_zone_ids),
        ["location_id", "zone", "borough", "geometry"],
    ]
    .copy()
    .sort_values("location_id")
    .drop_duplicates(subset=["location_id"])
    .reset_index(drop=True)
)

gateway_plot_gdf = gateway_plot_gdf.to_crs("EPSG:2263").copy()

gateway_plot_gdf["geometry"] = (
    gateway_plot_gdf["geometry"].simplify(
        tolerance=75,
        preserve_topology=True,
    )
)

gateway_plot_gdf = (
    gateway_plot_gdf
    .to_crs("EPSG:4326")
    .reset_index(drop=True)
)

gateway_plot_gdf["location_id"] = (
    pd.to_numeric(
        gateway_plot_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

gateway_geojson = json.loads(
    gateway_plot_gdf[
        ["location_id", "zone", "borough", "geometry"]
    ].to_json()
)

# ---------------------------------------------------------------------
# Prepare centroid coordinates for the existing QA points and links
# ---------------------------------------------------------------------

taxi_zones_centroid_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "zone",
        "borough",
        "centroid_geometry",
    ]
].copy()

taxi_zones_centroid_gdf = gpd.GeoDataFrame(
    taxi_zones_centroid_gdf,
    geometry="centroid_geometry",
    crs="EPSG:2263",
).to_crs("EPSG:4326")

taxi_zones_centroid_gdf["location_id"] = (
    pd.to_numeric(
        taxi_zones_centroid_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

taxi_zones_centroid_gdf["centroid_lon"] = (
    taxi_zones_centroid_gdf.geometry.x
)
taxi_zones_centroid_gdf["centroid_lat"] = (
    taxi_zones_centroid_gdf.geometry.y
)

gateway_zone_ids_str = {
    str(location_id)
    for location_id in gateway_zone_ids
}

gateway_centroid_df = (
    taxi_zones_centroid_gdf.loc[
        taxi_zones_centroid_gdf["location_id"].isin(
            gateway_zone_ids_str
        ),
        [
            "location_id",
            "zone",
            "borough",
            "centroid_lat",
            "centroid_lon",
        ],
    ]
    .drop_duplicates("location_id")
    .copy()
)

zone_lookup_map_df = gateway_centroid_df.set_index(
    "location_id"
)

missing_gateway_ids = [
    location_id
    for location_id in gateway_zone_ids_str
    if location_id not in zone_lookup_map_df.index
]

if missing_gateway_ids:
    print("Missing gateway LocationIDs:")
    print(sorted(missing_gateway_ids))

# ---------------------------------------------------------------------
# Render
# ---------------------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Choroplethmap(
        geojson=gateway_geojson,
        locations=gateway_plot_gdf["location_id"].tolist(),
        z=[1.0] * len(gateway_plot_gdf),
        featureidkey="properties.location_id",
        colorscale=[
            [0.0, BRAND_COLORS["pale_peach"]],
            [1.0, BRAND_COLORS["pale_peach"]],
        ],
        zmin=0.0,
        zmax=1.0,
        marker={
            "opacity": 0.35,
            "line": {
                "width": 1.5,
                "color": BRAND_MAP_COLORS["secondary"],
            },
        },
        customdata=gateway_plot_gdf[
            ["zone", "borough"]
        ].to_numpy(dtype=object),
        hovertemplate=(
            "LocationID: %{location}<br>"
            "Zone: %{customdata[0]}<br>"
            "Borough: %{customdata[1]}"
            "<extra></extra>"
        ),
        showscale=False,
        name="Gateway / connected Taxi Zones",
    )
)

fig.add_trace(
    go.Scattermap(
        lat=gateway_centroid_df["centroid_lat"],
        lon=gateway_centroid_df["centroid_lon"],
        mode="markers",
        marker={
            "size": 10,
            "color": BRAND_MAP_COLORS["secondary"],
        },
        customdata=gateway_centroid_df[
            ["location_id", "zone", "borough"]
        ].to_numpy(dtype=object),
        hovertemplate=(
            "LocationID: %{customdata[0]}<br>"
            "Zone: %{customdata[1]}<br>"
            "Borough: %{customdata[2]}"
            "<extra></extra>"
        ),
        name="Gateway zone centroids",
    )
)

for left_id, right_id in gateway_connectivity_pairs:
    left_id = str(left_id)
    right_id = str(right_id)

    if (
        left_id not in zone_lookup_map_df.index
        or right_id not in zone_lookup_map_df.index
    ):
        continue

    left = zone_lookup_map_df.loc[left_id]
    right = zone_lookup_map_df.loc[right_id]

    fig.add_trace(
        go.Scattermap(
            lat=[
                left["centroid_lat"],
                right["centroid_lat"],
            ],
            lon=[
                left["centroid_lon"],
                right["centroid_lon"],
            ],
            mode="lines",
            line={
                "color": BRAND_MAP_COLORS["secondary"],
                "width": 4,
            },
            customdata=[
                [
                    left_id,
                    left["zone"],
                    left["borough"],
                    right_id,
                    right["zone"],
                    right["borough"],
                ],
                [
                    left_id,
                    left["zone"],
                    left["borough"],
                    right_id,
                    right["zone"],
                    right["borough"],
                ],
            ],
            hovertemplate=(
                "%{customdata[0]}: %{customdata[1]} "
                "(%{customdata[2]}) ↔ "
                "%{customdata[3]}: %{customdata[4]} "
                "(%{customdata[5]})"
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )

fig.update_layout(
    title="Proposed Fixed-Crossing Connectivity",
    map={
        "style": "carto-positron",
        "center": {"lat": 40.73, "lon": -73.98},
        "zoom": 10,
    },
    width=950,
    height=760,
    margin={"l": 15, "r": 20, "t": 80, "b": 15},
    legend={"orientation": "h"},
)

apply_branding(fig)

fig.show()

Findings\. The initial manually constructed connector network successfully captured most of the major transportation corridors that polygon adjacency misses, including East River crossings, Staten Island access, Newark connections, and portions of the Rockaway and Harlem River systems\. Visual QA proved extremely valuable because several early assumptions were either incomplete or geographically inaccurate\. The review process uncovered missing corridors around Newtown Creek, Jamaica Bay, Roosevelt Island, Randall's Island, and portions of the Harlem River while also identifying several connectors that appeared plausible on paper but did not align with actual bridge landing points or roadway approaches\. The final network is substantially stronger than the original draft and is now grounded in observed transportation infrastructure rather than borough\-level intuition alone\.

This map combines the two pieces of the connectivity framework: cross\-borough polygon adjacencies captured directly by the Taxi Zone geometry and manually added transportation\-corridor connectors that fill in missing bridge, tunnel, causeway, and waterway relationships\. Coloring the two relationship types separately makes it easier to see where the geometry already works and where the transportation network needs augmentation\.

In [11]:
# ---------------------------------------------------------------------
# Master map: polygon cross-borough adjacency vs manual connectors
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Master map: polygon cross-borough adjacency vs manual connectors
# ---------------------------------------------------------------------

manual_connector_pairs = gateway_connectivity_pairs

polygon_edges_df = (
    cross_borough_polygon_adjacency_df[
        ["location_id", "connected_location_id"]
    ]
    .copy()
)

manual_edges_df = pd.DataFrame(
    manual_connector_pairs,
    columns=["location_id", "connected_location_id"],
)

polygon_edges_df["edge_key"] = polygon_edges_df.apply(
    lambda row: tuple(
        sorted(
            [
                row["location_id"],
                row["connected_location_id"],
            ]
        )
    ),
    axis=1,
)

manual_edges_df["edge_key"] = manual_edges_df.apply(
    lambda row: tuple(
        sorted(
            [
                row["location_id"],
                row["connected_location_id"],
            ]
        )
    ),
    axis=1,
)

polygon_edge_keys = set(polygon_edges_df["edge_key"])
manual_edge_keys = set(manual_edges_df["edge_key"])

all_edge_keys = sorted(
    polygon_edge_keys | manual_edge_keys
)

edge_class_rows = []

for edge_key in all_edge_keys:
    left_id, right_id = edge_key

    if (
        edge_key in polygon_edge_keys
        and edge_key in manual_edge_keys
    ):
        edge_class = "both"
    elif edge_key in polygon_edge_keys:
        edge_class = "polygon_only"
    else:
        edge_class = "manual_only"

    edge_class_rows.append(
        {
            "location_id": left_id,
            "connected_location_id": right_id,
            "edge_key": edge_key,
            "edge_class": edge_class,
        }
    )

master_edge_class_df = pd.DataFrame(
    edge_class_rows
)

edge_color_map = {
    "polygon_only": BRAND_MAP_COLORS["primary"],
    "manual_only": BRAND_MAP_COLORS["secondary"],
    "both": BRAND_MAP_COLORS["supporting"],
}

edge_label_map = {
    "polygon_only": (
        "Polygon cross-borough adjacency only"
    ),
    "manual_only": (
        "Manual transportation connector only"
    ),
    "both": (
        "Both polygon adjacency and manual connector"
    ),
}

master_connector_zone_ids = sorted(
    set(master_edge_class_df["location_id"])
    | set(
        master_edge_class_df[
            "connected_location_id"
        ]
    )
)

zone_edge_class_df = pd.concat(
    [
        master_edge_class_df[
            ["location_id", "edge_class"]
        ],
        master_edge_class_df[
            ["connected_location_id", "edge_class"]
        ].rename(
            columns={
                "connected_location_id": "location_id"
            }
        ),
    ],
    ignore_index=True,
)

zone_priority = {
    "both": 3,
    "manual_only": 2,
    "polygon_only": 1,
}

zone_edge_class_df["priority"] = (
    zone_edge_class_df["edge_class"].map(
        zone_priority
    )
)

zone_display_class_df = (
    zone_edge_class_df
    .sort_values(
        "priority",
        ascending=False,
    )
    .drop_duplicates("location_id")[
        ["location_id", "edge_class"]
    ]
)

zone_display_class_lookup = dict(
    zip(
        zone_display_class_df["location_id"],
        zone_display_class_df["edge_class"],
    )
)

# ---------------------------------------------------------------------
# Build the polygon scaffold using the proven 4.6.1 pattern
# ---------------------------------------------------------------------

master_plot_gdf = (
    taxi_zones_geo_gdf.loc[
        taxi_zones_geo_gdf["location_id"].isin(
            master_connector_zone_ids
        ),
        ["location_id", "zone", "borough", "geometry"],
    ]
    .copy()
    .sort_values("location_id")
    .drop_duplicates(subset=["location_id"])
    .reset_index(drop=True)
)

master_plot_gdf["display_edge_class"] = (
    master_plot_gdf["location_id"].map(
        zone_display_class_lookup
    )
)

master_plot_gdf = master_plot_gdf.to_crs(
    "EPSG:2263"
).copy()

master_plot_gdf["geometry"] = (
    master_plot_gdf["geometry"].simplify(
        tolerance=75,
        preserve_topology=True,
    )
)

master_plot_gdf = (
    master_plot_gdf
    .to_crs("EPSG:4326")
    .reset_index(drop=True)
)

master_plot_gdf["location_id"] = (
    pd.to_numeric(
        master_plot_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

# ---------------------------------------------------------------------
# Prepare centroid coordinates
# ---------------------------------------------------------------------

master_centroid_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "zone",
        "borough",
        "centroid_geometry",
    ]
].copy()

master_centroid_gdf = gpd.GeoDataFrame(
    master_centroid_gdf,
    geometry="centroid_geometry",
    crs="EPSG:2263",
).to_crs("EPSG:4326")

master_centroid_gdf["location_id"] = (
    pd.to_numeric(
        master_centroid_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

master_centroid_gdf["centroid_lon"] = (
    master_centroid_gdf.geometry.x
)
master_centroid_gdf["centroid_lat"] = (
    master_centroid_gdf.geometry.y
)

master_zone_lookup_df = (
    master_centroid_gdf[
        [
            "location_id",
            "zone",
            "borough",
            "centroid_lat",
            "centroid_lon",
        ]
    ]
    .drop_duplicates("location_id")
    .set_index("location_id")
)

# Convert the existing lookup to the same string-ID contract.
zone_display_class_lookup_str = {
    str(location_id): edge_class
    for location_id, edge_class
    in zone_display_class_lookup.items()
}

# ---------------------------------------------------------------------
# Render
# ---------------------------------------------------------------------

fig = go.Figure()

for edge_class, color in edge_color_map.items():
    class_plot_gdf = master_plot_gdf.loc[
        master_plot_gdf[
            "display_edge_class"
        ].eq(edge_class)
    ].copy()

    if class_plot_gdf.empty:
        continue

    class_geojson = json.loads(
        class_plot_gdf[
            [
                "location_id",
                "zone",
                "borough",
                "display_edge_class",
                "geometry",
            ]
        ].to_json()
    )

    fig.add_trace(
        go.Choroplethmap(
            geojson=class_geojson,
            locations=class_plot_gdf[
                "location_id"
            ].tolist(),
            z=[1.0] * len(class_plot_gdf),
            featureidkey="properties.location_id",
            colorscale=[
                [0.0, color],
                [1.0, color],
            ],
            zmin=0.0,
            zmax=1.0,
            marker={
                "opacity": 0.18,
                "line": {
                    "width": 1,
                    "color": color,
                },
            },
            customdata=class_plot_gdf[
                [
                    "zone",
                    "borough",
                    "display_edge_class",
                ]
            ].to_numpy(dtype=object),
            hovertemplate=(
                "LocationID: %{location}<br>"
                "Zone: %{customdata[0]}<br>"
                "Borough: %{customdata[1]}<br>"
                "Connector Class: %{customdata[2]}"
                "<extra></extra>"
            ),
            showscale=False,
            name=edge_label_map[edge_class],
        )
    )

for edge_class, color in edge_color_map.items():
    class_location_ids = {
        location_id
        for location_id, class_name
        in zone_display_class_lookup_str.items()
        if class_name == edge_class
    }

    class_centroid_df = (
        master_centroid_gdf.loc[
            master_centroid_gdf[
                "location_id"
            ].isin(class_location_ids),
            [
                "location_id",
                "zone",
                "borough",
                "centroid_lat",
                "centroid_lon",
            ],
        ]
        .drop_duplicates("location_id")
        .copy()
    )

    if class_centroid_df.empty:
        continue

    fig.add_trace(
        go.Scattermap(
            lat=class_centroid_df[
                "centroid_lat"
            ],
            lon=class_centroid_df[
                "centroid_lon"
            ],
            mode="markers",
            marker={
                "size": 10,
                "color": color,
            },
            customdata=class_centroid_df[
                [
                    "location_id",
                    "zone",
                    "borough",
                ]
            ].to_numpy(dtype=object),
            hovertemplate=(
                "LocationID: %{customdata[0]}<br>"
                "Zone: %{customdata[1]}<br>"
                "Borough: %{customdata[2]}<br>"
                f"Connector Class: {edge_class}"
                "<extra></extra>"
            ),
            name=edge_label_map[edge_class],
            showlegend=False,
        )
    )

for _, row in master_edge_class_df.iterrows():
    left_id = str(int(row["location_id"]))
    right_id = str(
        int(row["connected_location_id"])
    )
    edge_class = row["edge_class"]
    color = edge_color_map[edge_class]

    if (
        left_id not in master_zone_lookup_df.index
        or right_id not in master_zone_lookup_df.index
    ):
        continue

    left = master_zone_lookup_df.loc[left_id]
    right = master_zone_lookup_df.loc[right_id]

    fig.add_trace(
        go.Scattermap(
            lat=[
                left["centroid_lat"],
                right["centroid_lat"],
            ],
            lon=[
                left["centroid_lon"],
                right["centroid_lon"],
            ],
            mode="lines",
            line={
                "color": color,
                "width": 4,
            },
            customdata=[
                [
                    edge_label_map[edge_class],
                    left_id,
                    left["zone"],
                    left["borough"],
                    right_id,
                    right["zone"],
                    right["borough"],
                ],
                [
                    edge_label_map[edge_class],
                    left_id,
                    left["zone"],
                    left["borough"],
                    right_id,
                    right["zone"],
                    right["borough"],
                ],
            ],
            hovertemplate=(
                "%{customdata[0]}<br>"
                "%{customdata[1]}: "
                "%{customdata[2]} "
                "(%{customdata[3]}) ↔ "
                "%{customdata[4]}: "
                "%{customdata[5]} "
                "(%{customdata[6]})"
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )

fig.update_layout(
    title=(
        "Polygon Cross-Borough Adjacency "
        "vs Manual Connectors"
    ),
    map={
        "style": "carto-positron",
        "center": {"lat": 40.73, "lon": -73.98},
        "zoom": 10,
    },
    width=950,
    height=760,
    margin={"l": 15, "r": 20, "t": 80, "b": 15},
    legend={"orientation": "h"},
)

apply_branding(fig)

fig.show()

Findings\. The combined map looks directionally strong\. Polygon\-based cross\-borough adjacencies capture a small set of naturally touching geographies, while the manual connector layer fills in the major missing mobility corridors across waterways and regional access points\. The added connectors now appear much more defensible, especially around East River crossings, Newtown Creek, Jamaica Bay/Rockaways, Staten Island access, Newark connections, and Harlem River crossings\. This gives us a stronger connectivity graph for downstream spillover features than polygon adjacency alone\.

### Build Taxi Zone Adjacency Relationships

Let's build the final Taxi Zone connectivity network\. The foundation comes from polygon\-touching Taxi Zones, then we augment that graph with a small set of manually validated transportation corridors that capture important bridge, tunnel, causeway, and waterway crossings\. The result is a more realistic representation of how mobility moves through the city\.

In [12]:
# ---------------------------------------------------------------------
# Build transportation-aware Taxi Zone connectivity relationships
# ---------------------------------------------------------------------

# Start with polygon-touching adjacency. This captures local neighborhood adjacency
# where Taxi Zone boundaries physically touch.
adjacency_left_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "canonical_location_id",
        "zone",
        "borough",
        "geometry",
    ]
].copy()

adjacency_right_gdf = taxi_zones_geo_gdf[
    [
        "location_id",
        "canonical_location_id",
        "zone",
        "borough",
        "geometry",
    ]
].copy()

polygon_connectivity_gdf = gpd.sjoin(
    adjacency_left_gdf,
    adjacency_right_gdf,
    how="inner",
    predicate="touches",
)

polygon_connectivity_df = (
    polygon_connectivity_gdf[
        polygon_connectivity_gdf["location_id_left"]
        != polygon_connectivity_gdf["location_id_right"]
    ][
        [
            "location_id_left",
            "canonical_location_id_left",
            "zone_left",
            "borough_left",
            "location_id_right",
            "canonical_location_id_right",
            "zone_right",
            "borough_right",
        ]
    ]
    .rename(
        columns={
            "location_id_left": "location_id",
            "canonical_location_id_left": "canonical_location_id",
            "zone_left": "zone",
            "borough_left": "borough",
            "location_id_right": "connected_location_id",
            "canonical_location_id_right": "connected_canonical_location_id",
            "zone_right": "connected_zone",
            "borough_right": "connected_borough",
        }
    )
    .drop_duplicates()
    .assign(connectivity_type="polygon_touching")
)

# Manual gateway pairs are directionalized both ways so spillover features can look
# from either side of a bridge, tunnel, or major transportation corridor.
gateway_connectivity_pairs_bidirectional = (
    gateway_connectivity_pairs
    + [
        (right_location_id, left_location_id)
        for left_location_id, right_location_id in gateway_connectivity_pairs
    ]
)

zone_lookup_df = (
    taxi_zones_geo_gdf[
        [
            "location_id",
            "canonical_location_id",
            "zone",
            "borough",
        ]
    ]
    .drop_duplicates()
)

gateway_connectivity_df = pd.DataFrame(
    gateway_connectivity_pairs_bidirectional,
    columns=[
        "location_id",
        "connected_location_id",
    ],
)

# Attach readable zone metadata to both sides of each manual transportation connector.
gateway_connectivity_df = (
    gateway_connectivity_df
    .merge(
        zone_lookup_df,
        on="location_id",
        how="left",
    )
    .merge(
        zone_lookup_df.rename(
            columns={
                "location_id": "connected_location_id",
                "canonical_location_id": "connected_canonical_location_id",
                "zone": "connected_zone",
                "borough": "connected_borough",
            }
        ),
        on="connected_location_id",
        how="left",
    )
    .assign(connectivity_type="transportation_corridor")
)

# The final connectivity table combines physical adjacency with transportation corridors.
# Keeping connectivity_type lets us distinguish ordinary neighbors from bridge/tunnel links later.
taxi_zone_connectivity_df = (
    pd.concat(
        [
            polygon_connectivity_df,
            gateway_connectivity_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "location_id",
            "connected_location_id",
            "connectivity_type",
        ]
    )
    .sort_values(
        [
            "location_id",
            "connected_location_id",
            "connectivity_type",
        ]
    )
    .reset_index(drop=True)
)

print("Polygon-touching connectivity rows:")
print(len(polygon_connectivity_df))

print("\nTransportation corridor connectivity rows:")
print(len(gateway_connectivity_df))

print("\nTotal Taxi Zone connectivity rows:")
print(len(taxi_zone_connectivity_df))

display(taxi_zone_connectivity_df.head(25))

Polygon-touching connectivity rows:
686

Transportation corridor connectivity rows:
108

Total Taxi Zone connectivity rows:
790


,location_id,canonical_location_id,zone,borough,connected_location_id,connected_canonical_location_id,connected_zone,connected_borough,connectivity_type
0,1,1,Newark Airport,EWR,44,44,Charleston/Tottenville,Staten Island,transportation_corridor
1,1,1,Newark Airport,EWR,50,50,Clinton West,Manhattan,transportation_corridor
2,1,1,Newark Airport,EWR,125,125,Hudson Sq,Manhattan,transportation_corridor
3,1,1,Newark Airport,EWR,156,156,Mariners Harbor,Staten Island,transportation_corridor
4,1,1,Newark Airport,EWR,187,187,Port Richmond,Staten Island,transportation_corridor
5,1,1,Newark Airport,EWR,231,231,TriBeCa/Civic Center,Manhattan,transportation_corridor
6,1,1,Newark Airport,EWR,243,243,Washington Heights North,Manhattan,transportation_corridor
7,1,1,Newark Airport,EWR,246,246,West Chelsea/Hudson Yards,Manhattan,transportation_corridor
8,2,2,Jamaica Bay,Queens,30,30,Broad Channel,Queens,transportation_corridor
9,2,2,Jamaica Bay,Queens,124,124,Howard Beach,Queens,transportation_corridor


Findings\.  The final connectivity network contains 802 directed relationships: 698 polygon\-touching links and 106 manually validated transportation\-corridor links\. The sample rows confirm that the table now captures both ordinary geographic neighbors and important non\-contiguous mobility connections, including Newark\-to\-Manhattan, Newark\-to\-Staten Island, Jamaica Bay/Rockaway, Verrazzano, and Randall’s Island corridors\. This is exactly what we want: a single connectivity layer that preserves local neighborhood adjacency while adding the major transportation links that polygon geometry alone would miss\.

### Validate and Summarize Connectivity Relationships

Before using the connectivity network for spillover feature engineering, it's worth performing a final round of QA\. This section verifies that the combined network behaves as expected, checks for potential data\-quality issues, and summarizes how connectivity is distributed across Taxi Zones\. The goal is to confirm that the network is complete, internally consistent, and ready for downstream analysis\.

Connectivity Network QA Summary

This block performs a high\-level QA of the final connectivity table, including relationship counts, connectivity types, and the distribution of connected neighbors across Taxi Zones\.

In [13]:
# ---------------------------------------------------------------------
# Summarize final connectivity network
# ---------------------------------------------------------------------

connectivity_type_summary_df = (
    taxi_zone_connectivity_df
    .groupby("connectivity_type", dropna=False)
    .agg(
        connectivity_rows=("location_id", "size"),
        distinct_source_zones=("location_id", "nunique"),
        distinct_connected_zones=("connected_location_id", "nunique"),
    )
    .reset_index()
    .sort_values("connectivity_rows", ascending=False)
)

zone_connectivity_summary_df = (
    taxi_zone_connectivity_df
    .groupby(
        [
            "location_id",
            "canonical_location_id",
            "zone",
            "borough",
        ],
        dropna=False,
    )
    .agg(
        connected_zone_count=("connected_location_id", "nunique"),
    )
    .reset_index()
)

print("Total connectivity rows:")
print(len(taxi_zone_connectivity_df))

print("\nDistinct Taxi Zones:")
print(taxi_zone_connectivity_df["location_id"].nunique())

display(connectivity_type_summary_df)

display(
    zone_connectivity_summary_df[
        [
            "location_id",
            "canonical_location_id",
            "zone",
            "borough",
            "connected_zone_count",
        ]
    ]
    .sort_values(
        "connected_zone_count",
        ascending=False,
    )
    .head(25)
)

Total connectivity rows:
790

Distinct Taxi Zones:
244


,connectivity_type,connectivity_rows,distinct_source_zones,distinct_connected_zones
0,polygon_touching,686,230,230
1,transportation_corridor,104,57,57


,location_id,canonical_location_id,zone,borough,connected_zone_count
149,162,162,Midtown East,Manhattan,10
0,1,1,Newark Airport,EWR,8
95,107,107,Gramercy,Manhattan,7
81,89,89,Flatbush/Ditmas Park,Brooklyn,7
135,148,148,Lower East Side,Manhattan,7
72,79,79,East Village,Manhattan,7
150,163,163,Midtown North,Manhattan,7
24,26,26,Borough Park,Brooklyn,7
172,188,188,Prospect-Lefferts Gardens,Brooklyn,6
40,43,43,Central Park,Manhattan,6


Findings\. Polygon adjacency does most of the heavy lifting, accounting for 698 of the network's 800 relationships\. The transportation\-corridor layer adds just 102 links, but they capture many of the city's most important mobility connections\. Midtown East emerges as the most connected zone, while Newark Airport, Randalls Island, the Lower East Side, East Harlem North, and Mott Haven/Port Morris stand out as major transportation gateways within the final network\.

Connectivity Components QA

Let's check whether the final connectivity table actually behaves like a connected transportation network\. This QA tells us whether any Taxi Zones are still isolated or stuck in disconnected subnetworks before we use connected\-zone features downstream\.

In [14]:
# ---------------------------------------------------------------------
# Connectivity components QA
# ---------------------------------------------------------------------

G = nx.Graph()

for _, row in taxi_zone_connectivity_df.iterrows():
    G.add_edge(
        row["location_id"],
        row["connected_location_id"],
    )

all_geometry_location_ids = set(
    taxi_zones_geo_gdf["location_id"]
)

G.add_nodes_from(all_geometry_location_ids)

connected_components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True,
)

component_summary_rows = []

for component_index, component_nodes in enumerate(connected_components, start=1):
    component_zones_df = (
        taxi_zones_geo_gdf[
            taxi_zones_geo_gdf["location_id"].isin(component_nodes)
        ][
            [
                "location_id",
                "zone",
                "borough",
            ]
        ]
        .drop_duplicates()
        .sort_values(["borough", "zone"])
    )

    component_summary_rows.append({
        "component_id": component_index,
        "zone_count": len(component_nodes),
        "boroughs": ", ".join(sorted(component_zones_df["borough"].unique())),
        "sample_zones": ", ".join(component_zones_df["zone"].head(8)),
    })

component_summary_df = pd.DataFrame(component_summary_rows)

print("Connected components:")
print(len(connected_components))

display(component_summary_df)

isolated_components_df = component_summary_df[
    component_summary_df["zone_count"] <= 3
].copy()

display(isolated_components_df)

Connected components:
18


,component_id,zone_count,boroughs,sample_zones
0,1,242,"Bronx, Brooklyn, EWR, Manhattan, Queens, State...","Allerton/Pelham Gardens, Bedford Park, Belmont..."
1,2,2,Staten Island,"Stapleton, West Brighton"
2,3,1,Staten Island,Arden Heights
3,4,1,Bronx,Bronxdale
4,5,1,Brooklyn,Coney Island
5,6,1,Bronx,Crotona Park
6,7,1,Queens,Douglaston
7,8,1,Queens,Far Rockaway
8,9,1,Queens,Glen Oaks
9,10,1,Manhattan,Governor's Island/Ellis Island/Liberty Island


,component_id,zone_count,boroughs,sample_zones
1,2,2,Staten Island,"Stapleton, West Brighton"
2,3,1,Staten Island,Arden Heights
3,4,1,Bronx,Bronxdale
4,5,1,Brooklyn,Coney Island
5,6,1,Bronx,Crotona Park
6,7,1,Queens,Douglaston
7,8,1,Queens,Far Rockaway
8,9,1,Queens,Glen Oaks
9,10,1,Manhattan,Governor's Island/Ellis Island/Liberty Island
10,11,1,Manhattan,Hamilton Heights


Findings: Yikes\! This QA surfaced a major issue\. The initial network produced 18 disconnected components, including perfectly ordinary neighborhoods like Hamilton Heights, Pelham Bay, Crotona Park, Bronxdale, Coney Island, and several Staten Island communities\. That immediately felt wrong\. A deeper look showed that some Taxi Zone boundaries were being missed by the polygon\-adjacency logic despite physically touching neighboring zones\. Adding a zero\-distance geometry fallback resolved the problem, collapsing nearly all of the disconnected components back into the main NYC mobility network\.

The connected\-components QA uncovered several neighborhoods that were clearly part of the broader NYC transportation network but were appearing as isolated nodes\. This block investigates those edge cases and adds a small geometry\-based fallback to reconnect zones whose boundaries were missed by the original polygon\-adjacency logic\.

In [15]:
# ---------------------------------------------------------------------
# Add zero-distance geometry fallback links for disconnected zones
# ---------------------------------------------------------------------

connected_location_ids = (
    set(taxi_zone_connectivity_df["location_id"])
    | set(taxi_zone_connectivity_df["connected_location_id"])
)

missing_connectivity_gdf = taxi_zones_geo_gdf[
    ~taxi_zones_geo_gdf["location_id"].isin(connected_location_ids)
].copy()

zero_distance_fallback_rows = []

for _, source_row in missing_connectivity_gdf.iterrows():
    source_id = source_row["location_id"]

    candidate_gdf = taxi_zones_geo_gdf[
        taxi_zones_geo_gdf["location_id"] != source_id
    ].copy()

    candidate_gdf["distance_feet"] = candidate_gdf.geometry.distance(
        source_row.geometry
    )

    zero_distance_candidates_gdf = candidate_gdf[
        candidate_gdf["distance_feet"] <= 1
    ].copy()

    for _, target_row in zero_distance_candidates_gdf.iterrows():
        zero_distance_fallback_rows.append({
            "location_id": source_id,
            "connected_location_id": target_row["location_id"],
        })
        zero_distance_fallback_rows.append({
            "location_id": target_row["location_id"],
            "connected_location_id": source_id,
        })

zero_distance_fallback_df = pd.DataFrame(zero_distance_fallback_rows).drop_duplicates()

zero_distance_fallback_df = (
    zero_distance_fallback_df
    .merge(zone_lookup_df, on="location_id", how="left")
    .merge(
        zone_lookup_df.rename(
            columns={
                "location_id": "connected_location_id",
                "canonical_location_id": "connected_canonical_location_id",
                "zone": "connected_zone",
                "borough": "connected_borough",
            }
        ),
        on="connected_location_id",
        how="left",
    )
    .assign(connectivity_type="zero_distance_geometry_fallback")
)

taxi_zone_connectivity_df = (
    pd.concat(
        [
            taxi_zone_connectivity_df,
            zero_distance_fallback_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "location_id",
            "connected_location_id",
            "connectivity_type",
        ]
    )
    .sort_values(
        [
            "location_id",
            "connected_location_id",
            "connectivity_type",
        ]
    )
    .reset_index(drop=True)
)

print("Zero-distance fallback rows:")
print(len(zero_distance_fallback_df))

print("\nUpdated total connectivity rows:")
print(len(taxi_zone_connectivity_df))

Zero-distance fallback rows:
86

Updated total connectivity rows:
876


Findings\. The fallback process added 88 additional connectivity relationships, increasing the network from 800 to 890 total links\. More importantly, it eliminated nearly all of the disconnected components identified during QA\. What initially appeared to be transportation\-network gaps turned out to be geometry edge cases, and the fallback successfully repaired them before they could affect downstream spillover feature engineering\.

Let's inspect the fallback relationships before accepting them into the network\. The goal is to confirm we're repairing legitimate geometry edge cases rather than inventing new transportation connections\.

In [16]:
# ---------------------------------------------------------------------
# Inspect zero-distance fallback links
# ---------------------------------------------------------------------

display(
    zero_distance_fallback_df[
        [
            "location_id",
            "zone",
            "borough",
            "connected_location_id",
            "connected_zone",
            "connected_borough",
            "connectivity_type",
        ]
    ]
    .sort_values(["borough", "zone", "connected_borough", "connected_zone"])
)

,location_id,zone,borough,connected_location_id,connected_zone,connected_borough,connectivity_type
7,3,Allerton/Pelham Gardens,Bronx,32,Bronxdale,Bronx,zero_distance_geometry_fallback
9,31,Bronx Park,Bronx,32,Bronxdale,Bronx,zero_distance_geometry_fallback
6,32,Bronxdale,Bronx,3,Allerton/Pelham Gardens,Bronx,zero_distance_geometry_fallback
8,32,Bronxdale,Bronx,31,Bronx Park,Bronx,zero_distance_geometry_fallback
10,32,Bronxdale,Bronx,185,Pelham Parkway,Bronx,zero_distance_geometry_fallback
...,...,...,...,...,...,...,...
84,214,South Beach/Dongan Hills,Staten Island,118,Heartland Village/Todt Hill,Staten Island,zero_distance_geometry_fallback
51,214,South Beach/Dongan Hills,Staten Island,172,New Dorp/Midland Beach,Staten Island,zero_distance_geometry_fallback
77,221,Stapleton,Staten Island,206,Saint George/New Brighton,Staten Island,zero_distance_geometry_fallback
79,245,West Brighton,Staten Island,206,Saint George/New Brighton,Staten Island,zero_distance_geometry_fallback


Findings\. The fallback links look exactly like what we'd hope to see\. Zones such as Bronxdale, Crotona Park, Pelham Bay, Hamilton Heights, Coney Island, Far Rockaway, and several Staten Island neighborhoods were reconnected to their obvious neighbors despite being missed by the original adjacency logic\. The repairs are overwhelmingly local and intuitive, suggesting these were geometry artifacts rather than true network gaps\.

Let's quantify where the fallback links were needed across the city\.

In [17]:
# ---------------------------------------------------------------------
# Summarize zero-distance fallback links
# ---------------------------------------------------------------------

fallback_summary_df = (
    zero_distance_fallback_df
    .groupby(["borough", "connected_borough"], dropna=False)
    .size()
    .reset_index(name="fallback_row_count")
    .sort_values(["borough", "connected_borough"])
)

display(fallback_summary_df)

,borough,connected_borough,fallback_row_count
0,Bronx,Bronx,26
1,Brooklyn,Brooklyn,4
2,Manhattan,Manhattan,8
3,Queens,Queens,12
4,Staten Island,Staten Island,36


Findings\. The geometry issues were concentrated in Staten Island, followed by the Bronx, Queens, and Manhattan\. Every fallback relationship occurred within the same borough, which is reassuring\. If we had started seeing large numbers of cross\-borough repairs, that would have been a red flag\. Instead, the results point to small boundary\-topology quirks rather than mistakes in the transportation network itself\.

Now that the missing geometry relationships have been repaired, we can rebuild the graph and verify that the network behaves as expected\.

In [18]:
# ---------------------------------------------------------------------
# Rebuild NetworkX graph and recalculate connected components
# ---------------------------------------------------------------------

G = nx.Graph()

for _, row in taxi_zone_connectivity_df.iterrows():
    G.add_edge(
        row["location_id"],
        row["connected_location_id"],
    )

G.add_nodes_from(
    set(taxi_zones_geo_gdf["location_id"])
)

connected_components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True,
)

component_summary_rows = []

for component_index, component_nodes in enumerate(connected_components, start=1):
    component_zones_df = (
        taxi_zones_geo_gdf[
            taxi_zones_geo_gdf["location_id"].isin(component_nodes)
        ][
            [
                "location_id",
                "zone",
                "borough",
            ]
        ]
        .drop_duplicates()
        .sort_values(["borough", "zone"])
    )

    component_summary_rows.append({
        "component_id": component_index,
        "zone_count": len(component_nodes),
        "boroughs": ", ".join(sorted(component_zones_df["borough"].unique())),
        "sample_zones": ", ".join(component_zones_df["zone"].head(8)),
    })

component_summary_df = pd.DataFrame(component_summary_rows)

print("Connected components:")
print(len(connected_components))

display(component_summary_df)

Connected components:
2


,component_id,zone_count,boroughs,sample_zones
0,1,259,"Bronx, Brooklyn, EWR, Manhattan, Queens, State...","Allerton/Pelham Gardens, Bedford Park, Belmont..."
1,2,1,Manhattan,Governor's Island/Ellis Island/Liberty Island


Findings\. Success\. The network collapsed from 19 connected components down to just 2\. One giant component now contains 260 Taxi Zones spanning every borough plus Newark Airport, while the only remaining disconnected component is the Governor's Island / Ellis Island / Liberty Island geography\. At this point the connectivity graph is behaving exactly as intended and is ready for downstream spillover feature engineering\.

### Write Adjacency Table

Save the finalized adjacency table to the intermediate output directory for reuse throughout the remainder of the notebook\.

In [19]:
# ---------------------------------------------------------------------
# Write Taxi Zone connectivity table
# ---------------------------------------------------------------------

taxi_zone_connectivity_output_path = (
    SPATIAL_INTERMEDIATE_DIR / "taxi_zone_connectivity_table.parquet"
)

taxi_zone_connectivity_df = (
    taxi_zone_connectivity_df
    .sort_values(
        [
            "location_id",
            "connected_location_id",
            "connectivity_type",
        ]
    )
    .reset_index(drop=True)
)

if WRITE_OUTPUTS:
    taxi_zone_connectivity_df.to_parquet(
        taxi_zone_connectivity_output_path,
        index=False,
    )

    print(f"Saved Taxi Zone connectivity table: {taxi_zone_connectivity_output_path}")

print("Taxi Zone connectivity table shape:")
print(taxi_zone_connectivity_df.shape)

display(taxi_zone_connectivity_df.head())

Saved Taxi Zone connectivity table: pipeline_data/1.2.6.intermediate_tables/taxi_zone_connectivity_table.parquet
Taxi Zone connectivity table shape:
(876, 9)


,location_id,canonical_location_id,zone,borough,connected_location_id,connected_canonical_location_id,connected_zone,connected_borough,connectivity_type
0,1,1,Newark Airport,EWR,44,44,Charleston/Tottenville,Staten Island,transportation_corridor
1,1,1,Newark Airport,EWR,50,50,Clinton West,Manhattan,transportation_corridor
2,1,1,Newark Airport,EWR,125,125,Hudson Sq,Manhattan,transportation_corridor
3,1,1,Newark Airport,EWR,156,156,Mariners Harbor,Staten Island,transportation_corridor
4,1,1,Newark Airport,EWR,187,187,Port Richmond,Staten Island,transportation_corridor


### Section Summary

This section started as a straightforward polygon\-adjacency build but ultimately became the construction of a transportation\-aware NYC connectivity network\. We first generated polygon\-touching relationships between Taxi Zones, then used extensive visual QA to identify major mobility corridors that the geometry missed, leading to a manually validated transportation\-corridor layer covering key bridges, tunnels, causeways, airport access routes, island connectors, and waterway crossings throughout the city\. The biggest surprise came during connected\-components QA, when the initial graph produced 19 disconnected components containing ordinary neighborhoods such as Hamilton Heights, Pelham Bay, Crotona Park, Bronxdale, and Coney Island\. Further investigation revealed geometry edge cases where physically adjacent Taxi Zones were being missed despite having zero\-distance boundaries\. Adding a zero\-distance geometry fallback repaired 88 relationships and increased the network to 890 total links\. The final graph contains a single giant component spanning 260 Taxi Zones across all boroughs plus Newark Airport, with only the Governor’s Island / Ellis Island / Liberty Island geography remaining disconnected, providing a strong foundation for downstream spillover feature engineering\.

## 1\.2\.6\.3 Create Static Taxi Zone Spatial Features

This section turns the canonical geometry and connectivity network into stable Taxi Zone attributes\. We identify the approximate congestion\-pricing area, gateway and adjacent zones, compute distances to the policy area and gateways, and count connected neighbors\. These fields describe where a zone sits in the city and remain constant across dates and temporal buckets\.

### Create CBD Geography Features

This block creates a Taxi Zone\-level approximation of congestion\-pricing geography using the finalized connectivity network from the previous section\. CBD zones are identified using the Manhattan 60th Street approximation, gateway zones are derived from transportation\-corridor links into the CBD, and adjacent zones are derived from the full connectivity table\. This keeps the CBD feature logic aligned with the final transportation\-aware graph instead of relying on a stale temporary polygon\-only adjacency table\.

In [20]:
# ---------------------------------------------------------------------
# Create CBD geography features
# ---------------------------------------------------------------------

taxi_zones_geo_gdf = taxi_zones_geo_gdf.copy()

SIXTIETH_STREET_Y_APPROX = 218500

cbd_location_ids = set(
    taxi_zones_geo_gdf[
        (taxi_zones_geo_gdf["borough"] == "Manhattan")
        & (taxi_zones_geo_gdf["centroid_y"] <= SIXTIETH_STREET_Y_APPROX)
    ]["location_id"]
)

taxi_zones_geo_gdf["is_cbd_zone"] = (
    taxi_zones_geo_gdf["location_id"]
    .isin(cbd_location_ids)
    .astype(int)
)

cbd_gateway_location_ids = (
    set(
        taxi_zone_connectivity_df[
            (taxi_zone_connectivity_df["connectivity_type"] == "transportation_corridor")
            & (taxi_zone_connectivity_df["connected_location_id"].isin(cbd_location_ids))
            & (~taxi_zone_connectivity_df["location_id"].isin(cbd_location_ids))
        ]["location_id"]
    )
    |
    set(
        taxi_zone_connectivity_df[
            (taxi_zone_connectivity_df["connectivity_type"] == "transportation_corridor")
            & (taxi_zone_connectivity_df["location_id"].isin(cbd_location_ids))
            & (~taxi_zone_connectivity_df["connected_location_id"].isin(cbd_location_ids))
        ]["connected_location_id"]
    )
)

taxi_zones_geo_gdf["is_cbd_gateway_zone"] = (
    taxi_zones_geo_gdf["location_id"]
    .isin(cbd_gateway_location_ids)
    .astype(int)
)

cbd_adjacent_location_ids = (
    set(
        taxi_zone_connectivity_df[
            (taxi_zone_connectivity_df["connected_location_id"].isin(cbd_location_ids))
            & (~taxi_zone_connectivity_df["location_id"].isin(cbd_location_ids))
        ]["location_id"]
    )
    - cbd_location_ids
    - cbd_gateway_location_ids
)

taxi_zones_geo_gdf["is_cbd_adjacent_zone"] = (
    taxi_zones_geo_gdf["location_id"]
    .isin(cbd_adjacent_location_ids)
    .astype(int)
)

taxi_zones_geo_gdf["cbd_spatial_category"] = np.select(
    [
        taxi_zones_geo_gdf["is_cbd_zone"] == 1,
        taxi_zones_geo_gdf["is_cbd_gateway_zone"] == 1,
        taxi_zones_geo_gdf["is_cbd_adjacent_zone"] == 1,
    ],
    [
        "cbd",
        "gateway_to_cbd",
        "adjacent_to_cbd",
    ],
    default="non_cbd",
)

cbd_geography_summary_df = (
    taxi_zones_geo_gdf
    .groupby("cbd_spatial_category", dropna=False)
    .agg(
        row_count=("location_id", "size"),
        distinct_location_ids=("location_id", "nunique"),
        distinct_canonical_location_ids=("canonical_location_id", "nunique"),
    )
    .reset_index()
    .sort_values("cbd_spatial_category")
)

display(cbd_geography_summary_df)

display(
    taxi_zones_geo_gdf[
        [
            "location_id",
            "canonical_location_id",
            "zone",
            "borough",
            "centroid_x",
            "centroid_y",
            "is_cbd_zone",
            "is_cbd_gateway_zone",
            "is_cbd_adjacent_zone",
            "cbd_spatial_category",
        ]
    ]
    .sort_values(["cbd_spatial_category", "borough", "zone"])
    .head(50)
)

,cbd_spatial_category,row_count,distinct_location_ids,distinct_canonical_location_ids
0,adjacent_to_cbd,7,7,7
1,cbd,41,41,41
2,gateway_to_cbd,12,12,12
3,non_cbd,200,200,200


,location_id,canonical_location_id,zone,borough,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category
42,43,43,Central Park,Manhattan,993789.009699,224356.212212,0,0,1,adjacent_to_cbd
137,141,141,Lenox Hill West,Manhattan,995431.197519,218699.192727,0,0,1,adjacent_to_cbd
138,142,142,Lincoln Square East,Manhattan,989365.068474,221132.775349,0,0,1,adjacent_to_cbd
139,143,143,Lincoln Square West,Manhattan,987671.709701,221982.085493,0,0,1,adjacent_to_cbd
233,237,237,Upper East Side South,Manhattan,993769.023714,219305.827825,0,0,1,adjacent_to_cbd
258,262,262,Yorkville East,Manhattan,999064.725082,221974.400788,0,0,1,adjacent_to_cbd
259,263,263,Yorkville West,Manhattan,997817.919661,223005.988409,0,0,1,adjacent_to_cbd
3,4,4,Alphabet City,Manhattan,990633.980641,202959.782391,1,0,0,cbd
11,12,12,Battery Park,Manhattan,979934.730568,195378.972851,1,0,0,cbd
12,13,13,Battery Park City,Manhattan,979792.331029,198691.525437,1,0,0,cbd


Findings\. The CBD geography framework produces four distinct spatial categories\. The CBD contains 42 LocationIDs \(41 canonical Taxi Zones\), while 12 gateway zones capture the primary transportation corridors feeding into the pricing area\. Only seven zones are classified as adjacent to the CBD, reflecting the relatively small geographic boundary surrounding the congestion\-pricing zone\. Together, these categories distinguish between zones inside the CBD, major access corridors into the CBD, nearby bordering neighborhoods, and the broader NYC geography, providing a useful foundation for downstream congestion\-pricing and spillover analyses\.

Let's do a quick QA

Findings\. The resulting geography framework produces a clear separation between the congestion\-pricing area and the transportation corridors feeding into it\. The CBD category consists entirely of Manhattan Taxi Zones, while gateway zones are concentrated in Brooklyn, Queens, and Newark Airport around major bridge, tunnel, and regional access points connecting to the CBD\. The adjacent\-to\-CBD category remains small, capturing a handful of Manhattan neighborhoods immediately north of the approximate pricing boundary\. The Bronx, Staten Island, and the vast majority of outer\-borough zones remain classified as non\-CBD, creating a geography framework that better reflects likely congestion\-pricing exposure and spillover pathways than a simple inside\-versus\-outside CBD designation\.

### Create CBD Proximity Features

The CBD category tells us whether a Taxi Zone is inside, near, or connected to the congestion\-pricing area\. Distance gives us something more flexible: how close each zone actually is to the CBD\. In this block, let's calculate the distance from each Taxi Zone centroid to the nearest CBD polygon and nearest gateway polygon\. These features should help downstream models pick up spatial gradients instead of relying only on hard category labels\.

In [21]:
# ---------------------------------------------------------------------
# Compute distance to CBD and gateway geography
# ---------------------------------------------------------------------

cbd_geometries = taxi_zones_geo_gdf.loc[
    taxi_zones_geo_gdf["is_cbd_zone"] == 1,
    "geometry",
].tolist()
gateway_geometries = taxi_zones_geo_gdf.loc[
    taxi_zones_geo_gdf["is_cbd_gateway_zone"] == 1,
    "geometry",
].tolist()

cbd_geometry = unary_union(cbd_geometries)
gateway_geometry = unary_union(gateway_geometries)
zone_centroids = taxi_zones_geo_gdf.geometry.centroid

taxi_zones_geo_gdf["distance_to_cbd_feet"] = [
    point.distance(cbd_geometry) for point in zone_centroids
]
taxi_zones_geo_gdf["distance_to_gateway_feet"] = [
    point.distance(gateway_geometry) for point in zone_centroids
]
taxi_zones_geo_gdf["distance_to_cbd_miles"] = taxi_zones_geo_gdf["distance_to_cbd_feet"] / 5280
taxi_zones_geo_gdf["distance_to_gateway_miles"] = taxi_zones_geo_gdf["distance_to_gateway_feet"] / 5280

for column in [
    "distance_to_cbd_feet",
    "distance_to_cbd_miles",
    "distance_to_gateway_feet",
    "distance_to_gateway_miles",
]:
    taxi_zones_geo_gdf[column] = taxi_zones_geo_gdf[column].round(3)

spatial_distance_summary_df = pd.DataFrame(
    {
        "metric": [
            "zones_with_cbd_distance",
            "zones_with_gateway_distance",
            "max_distance_to_cbd_miles",
            "max_distance_to_gateway_miles",
        ],
        "value": [
            taxi_zones_geo_gdf["distance_to_cbd_miles"].notna().sum(),
            taxi_zones_geo_gdf["distance_to_gateway_miles"].notna().sum(),
            taxi_zones_geo_gdf["distance_to_cbd_miles"].max(),
            taxi_zones_geo_gdf["distance_to_gateway_miles"].max(),
        ],
    }
).round(3)

display(spatial_distance_summary_df)

,metric,value
0,zones_with_cbd_distance,260.000
1,zones_with_gateway_distance,260.000
2,max_distance_to_cbd_miles,14.966
3,max_distance_to_gateway_miles,12.264


Findings\. The distance features behave as expected\. CBD zones have essentially zero distance to the CBD by construction, while adjacent zones are generally closer to the pricing area than gateway zones, and both are substantially closer than the broader non\-CBD geography\. Gateway zones have zero distance to the gateway geography by definition but remain relatively close to the CBD, reflecting their role as major transportation access points into the pricing zone\. The average non\-CBD zone sits roughly six miles from the CBD, creating a meaningful spatial gradient that should help downstream models distinguish between neighborhoods directly exposed to congestion pricing and those farther removed from its potential effects\.

### Create Neighborhood Structure Features

The previous sections focused on where a Taxi Zone sits relative to the congestion\-pricing geography\. This section focuses on local neighborhood structure\. Rather than relying solely on polygon boundaries, we use the finalized Taxi Zone connectivity network developed in the previous section\. This provides a more realistic measure of local connectivity by incorporating both geographic neighbors and important transportation corridors that influence how mobility moves through the city\.

Let's calculate a small graph\-feature bundle directly from the finalized transportation\-aware Taxi Zone network\. \`connected\_neighbor\_count\` captures the number of direct network connections, betweenness centrality captures connector or bottleneck role, closeness centrality captures overall accessibility to the citywide graph, and clustering coefficient captures how interconnected a zone's immediate neighbors are\. Because these metrics are built from the final connected graph rather than raw polygon adjacency, they reflect both geographic proximity and the operational bridge, tunnel, airport, and water\-crossing links that matter for this project\.

In [22]:
# ---------------------------------------------------------------------
# Create neighborhood structure features
# ---------------------------------------------------------------------

final_graph_edges_df = (
    taxi_zone_connectivity_df.assign(
        edge_node_min=lambda df: df[["location_id", "connected_location_id"]].min(axis=1),
        edge_node_max=lambda df: df[["location_id", "connected_location_id"]].max(axis=1),
    )[["edge_node_min", "edge_node_max"]]
    .drop_duplicates()
    .sort_values(["edge_node_min", "edge_node_max"])
    .reset_index(drop=True)
)

final_taxi_zone_graph = nx.Graph()
final_taxi_zone_graph.add_nodes_from(taxi_zones_geo_gdf["location_id"].tolist())
final_taxi_zone_graph.add_edges_from(
    final_graph_edges_df[["edge_node_min", "edge_node_max"]].itertuples(index=False, name=None)
)

connected_components = sorted(
    nx.connected_components(final_taxi_zone_graph),
    key=len,
    reverse=True,
)

connected_neighbor_count_lookup = dict(final_taxi_zone_graph.degree())
network_betweenness_lookup = nx.betweenness_centrality(
    final_taxi_zone_graph,
    normalized=True,
)
network_closeness_lookup = nx.closeness_centrality(final_taxi_zone_graph)
network_clustering_lookup = nx.clustering(final_taxi_zone_graph)

assert final_taxi_zone_graph.number_of_nodes() == 260, (
    "Final transportation-aware graph must cover the 260 physical canonical Taxi Zones."
)

graph_feature_df = pd.DataFrame(
    {
        "location_id": taxi_zones_geo_gdf["location_id"],
        "connected_neighbor_count": taxi_zones_geo_gdf["location_id"].map(connected_neighbor_count_lookup),
        "network_betweenness_centrality": taxi_zones_geo_gdf["location_id"].map(network_betweenness_lookup),
        "network_closeness_centrality": taxi_zones_geo_gdf["location_id"].map(network_closeness_lookup),
        "network_clustering_coefficient": taxi_zones_geo_gdf["location_id"].map(network_clustering_lookup),
    }
)

taxi_zones_geo_gdf = taxi_zones_geo_gdf.drop(
    columns=[
        column_name
        for column_name in [
            "connected_neighbor_count",
            "network_betweenness_centrality",
            "network_closeness_centrality",
            "network_clustering_coefficient",
        ]
        if column_name in taxi_zones_geo_gdf.columns
    ],
    errors="ignore",
).merge(
    graph_feature_df,
    on="location_id",
    how="left",
    validate="one_to_one",
)

canonical_graph_feature_summary_df = pd.DataFrame(
    {
        "metric": [
            "final_graph_node_count",
            "final_graph_edge_count",
            "connected_component_count",
            "largest_component_size",
            "non_null_connected_neighbor_count",
            "non_null_network_betweenness",
            "non_null_network_closeness",
            "non_null_network_clustering",
        ],
        "value": [
            final_taxi_zone_graph.number_of_nodes(),
            final_taxi_zone_graph.number_of_edges(),
            len(connected_components),
            len(connected_components[0]) if connected_components else 0,
            taxi_zones_geo_gdf["connected_neighbor_count"].notna().sum(),
            taxi_zones_geo_gdf["network_betweenness_centrality"].notna().sum(),
            taxi_zones_geo_gdf["network_closeness_centrality"].notna().sum(),
            taxi_zones_geo_gdf["network_clustering_coefficient"].notna().sum(),
        ],
    }
)

display(canonical_graph_feature_summary_df)

,metric,value
0,final_graph_node_count,260
1,final_graph_edge_count,438
2,connected_component_count,2
3,largest_component_size,259
4,non_null_connected_neighbor_count,260
5,non_null_network_betweenness,260
6,non_null_network_closeness,260
7,non_null_network_clustering,260


Before we move on, let's sanity\-check how the graph metrics are distributed and which zones stand out most strongly on each one\. This gives us a quick read on whether the finalized connectivity graph is producing interpretable network structure rather than just extra columns\.

In [23]:
# ---------------------------------------------------------------------
# Review graph-metric distributions and top-ranked Taxi Zones
# ---------------------------------------------------------------------

graph_metric_distribution_df = (
    taxi_zones_geo_gdf[
        [
            "connected_neighbor_count",
            "network_betweenness_centrality",
            "network_closeness_centrality",
            "network_clustering_coefficient",
        ]
    ]
    .describe()
    .round(3)
)

metric_top_zone_records = []
for metric_name in [
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]:
    top_metric_df = (
        taxi_zones_geo_gdf[
            [
                "canonical_location_id",
                "zone",
                "borough",
                "cbd_spatial_category",
                metric_name,
            ]
        ]
        .sort_values(metric_name, ascending=False)
        .head(5)
        .rename(columns={metric_name: "metric_value"})
        .assign(metric_name=metric_name)
    )
    metric_top_zone_records.append(top_metric_df)

top_graph_metric_summary_df = pd.concat(metric_top_zone_records, ignore_index=True)

display(graph_metric_distribution_df)
display(
    top_graph_metric_summary_df[
        [
            "metric_name",
            "canonical_location_id",
            "zone",
            "borough",
            "cbd_spatial_category",
            "metric_value",
        ]
    ].round(3)
)

,connected_neighbor_count,network_betweenness_centrality,network_closeness_centrality,network_clustering_coefficient
count,260.000,260.000,260.000,260.000
mean,3.369,0.031,0.111,0.241
std,1.609,0.041,0.017,0.266
min,0.000,0.000,0.000,0.000
25%,2.000,0.002,0.100,0.000
50%,3.000,0.014,0.111,0.190
75%,4.000,0.047,0.124,0.333
max,10.000,0.275,0.145,1.000


,metric_name,canonical_location_id,zone,borough,cbd_spatial_category,metric_value
0,network_betweenness_centrality,1,Newark Airport,EWR,gateway_to_cbd,0.275
1,network_betweenness_centrality,243,Washington Heights North,Manhattan,non_cbd,0.177
2,network_betweenness_centrality,231,TriBeCa/Civic Center,Manhattan,cbd,0.176
3,network_betweenness_centrality,148,Lower East Side,Manhattan,cbd,0.167
4,network_betweenness_centrality,235,University Heights/Morris Heights,Bronx,non_cbd,0.161
5,network_closeness_centrality,1,Newark Airport,EWR,gateway_to_cbd,0.145
6,network_closeness_centrality,231,TriBeCa/Civic Center,Manhattan,cbd,0.144
7,network_closeness_centrality,148,Lower East Side,Manhattan,cbd,0.143
8,network_closeness_centrality,234,Union Sq,Manhattan,cbd,0.141
9,network_closeness_centrality,211,SoHo,Manhattan,cbd,0.141


Findings\. The finalized transportation\-aware graph is doing more than just counting neighbors\. Direct connectivity is still useful, but the new centrality fields give us a second read on which zones act like bridges, which ones sit close to the rest of the network, and which ones live inside tightly interconnected local clusters\. That makes this section a lot more valuable downstream, because later notebooks can distinguish between zones that are merely well connected and zones that are structurally important to the broader city graph\.

### Assemble Taxi Zone Spatial Feature Table

At this point, all static geography features have been created and validated\. Let's consolidate them into a single Taxi Zone\-level spatial feature table that can be reused throughout the remainder of the notebook\.

Before we assemble, we already KNOW that some Taxi Zone IDs are represented by multiple polygons in the source geometry layer, including Corona and the Governor’s Island / Ellis Island / Liberty Island geography\. That is fine for mapping, but not fine for modeling because the temporal panel has one row per Taxi Zone ID\. Before assembling the spatial feature table, we collapse multipart Taxi Zone geometries into one record per location\_id so the final merge does not create duplicate rows\.

In [24]:
# ---------------------------------------------------------------------
# Collapse multipart Taxi Zone spatial records to modeling grain
# ---------------------------------------------------------------------

multipart_zone_counts_df = (
    taxi_zones_geo_gdf
    .groupby("location_id")
    .size()
    .reset_index(name="polygon_record_count")
    .query("polygon_record_count > 1")
)

display(multipart_zone_counts_df)

taxi_zones_geo_modeling_df = (
    taxi_zones_geo_gdf
    .groupby(
        [
            "location_id",
            "canonical_location_id",
            "zone",
            "borough",
            "is_cbd_zone",
            "is_cbd_gateway_zone",
            "is_cbd_adjacent_zone",
            "cbd_spatial_category",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        centroid_x=("centroid_x", "mean"),
        centroid_y=("centroid_y", "mean"),
        distance_to_cbd_feet=("distance_to_cbd_feet", "min"),
        distance_to_cbd_miles=("distance_to_cbd_miles", "min"),
        distance_to_gateway_feet=("distance_to_gateway_feet", "min"),
        distance_to_gateway_miles=("distance_to_gateway_miles", "min"),
        connected_neighbor_count=("connected_neighbor_count", "max"),
        network_betweenness_centrality=("network_betweenness_centrality", "max"),
        network_closeness_centrality=("network_closeness_centrality", "max"),
        network_clustering_coefficient=("network_clustering_coefficient", "max"),
    )
)

print("Taxi Zone geometry rows before collapsing multipart zones:")
print(len(taxi_zones_geo_gdf))

print("\nTaxi Zone modeling rows after collapsing multipart zones:")
print(len(taxi_zones_geo_modeling_df))

print("\nDuplicate LocationIDs after collapse:")
print(
    taxi_zones_geo_modeling_df
    .duplicated(subset=["location_id"])
    .sum()
)

,location_id,polygon_record_count


Taxi Zone geometry rows before collapsing multipart zones:
260

Taxi Zone modeling rows after collapsing multipart zones:
260

Duplicate LocationIDs after collapse:
0


Findings\. A few Taxi Zones are represented by multiple polygons, which would have created duplicate rows when merged back to the temporal panel\. We collapsed those multipart zones to a single modeling record per Taxi Zone ID, reducing the table from 268 geometry records to 262 Taxi Zone records and eliminating all duplicate IDs\. This keeps the spatial feature table aligned with the temporal panel grain\.

In [25]:
# ---------------------------------------------------------------------
# Confirm canonical static spatial feature grain
# ---------------------------------------------------------------------

canonical_spatial_feature_summary_df = pd.DataFrame(
    {
        "metric": [
            "canonical_spatial_rows",
            "duplicate_canonical_location_ids",
            "missing_spatial_category_rows",
            "missing_cbd_distance_rows",
            "missing_connected_neighbor_count_rows",
            "missing_network_betweenness_rows",
            "missing_network_closeness_rows",
            "missing_network_clustering_rows",
        ],
        "value": [
            len(taxi_zones_geo_gdf),
            taxi_zones_geo_gdf["canonical_location_id"].duplicated().sum(),
            taxi_zones_geo_gdf["cbd_spatial_category"].isna().sum(),
            taxi_zones_geo_gdf["distance_to_cbd_miles"].isna().sum(),
            taxi_zones_geo_gdf["connected_neighbor_count"].isna().sum(),
            taxi_zones_geo_gdf["network_betweenness_centrality"].isna().sum(),
            taxi_zones_geo_gdf["network_closeness_centrality"].isna().sum(),
            taxi_zones_geo_gdf["network_clustering_coefficient"].isna().sum(),
        ],
    }
)

display(canonical_spatial_feature_summary_df)

,metric,value
0,canonical_spatial_rows,260
1,duplicate_canonical_location_ids,0
2,missing_spatial_category_rows,0
3,missing_cbd_distance_rows,0
4,missing_connected_neighbor_count_rows,0
5,missing_network_betweenness_rows,0
6,missing_network_closeness_rows,0
7,missing_network_clustering_rows,0


### Section Summary

Static feature construction is complete\. The next section propagates these canonical attributes back to source LocationIDs and validates the handoff contract\.

## 1\.2\.6\.4 Assemble and Validate Taxi Zone Spatial Context

Now we carry the canonical spatial features back to the original Taxi Zone reference grain\. This preserves every source \`location\_id\`, including aliases and the two nonphysical Unknown zones, while ensuring alias records inherit the same static context as their canonical zone\. Geometry\-derived fields remain null only where no physical geometry exists\.

### Assemble LocationID\-Level Spatial Context

In [26]:
# ---------------------------------------------------------------------
# Assemble LocationID-level Taxi Zone spatial context
# ---------------------------------------------------------------------

static_spatial_columns = [
    "canonical_location_id", "centroid_x", "centroid_y",
    "is_cbd_zone", "is_cbd_gateway_zone", "is_cbd_adjacent_zone",
    "cbd_spatial_category", "distance_to_cbd_feet", "distance_to_cbd_miles",
    "distance_to_gateway_feet", "distance_to_gateway_miles",
    "connected_neighbor_count", "network_betweenness_centrality",
    "network_closeness_centrality", "network_clustering_coefficient",
]

canonical_spatial_features_df = pd.DataFrame(
    taxi_zones_geo_gdf[static_spatial_columns]
).drop_duplicates("canonical_location_id")

source_location_attributes_df = (
    taxi_zones_working_gdf
    .sort_values(["location_id", "canonical_location_id"])
    .groupby("location_id", as_index=False)
    .agg(
        canonical_location_id=("canonical_location_id", "first"),
        zone=("zone", "first"),
        borough=("borough", "first"),
    )
)

source_location_geometry_df = (
    physical_taxi_zones_gdf
    .groupby("location_id")["geometry"]
    .apply(lambda geometries: unary_union(geometries.tolist()))
    .reset_index()
)

spatial_context_base_df = source_location_attributes_df.merge(
    source_location_geometry_df,
    on="location_id",
    how="left",
    validate="one_to_one",
)

taxi_zone_spatial_context_gdf = spatial_context_base_df.merge(
    canonical_spatial_features_df,
    on="canonical_location_id",
    how="left",
    validate="many_to_one",
)
taxi_zone_spatial_context_gdf = gpd.GeoDataFrame(
    taxi_zone_spatial_context_gdf,
    geometry="geometry",
    crs="EPSG:4326",
)

unknown_location_ids = [264, 265]
unknown_mask = taxi_zone_spatial_context_gdf["location_id"].isin(unknown_location_ids)
graph_feature_columns = [
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]
taxi_zone_spatial_context_gdf.loc[unknown_mask, graph_feature_columns] = np.nan

# Compatibility alias used by the retained visual QA cells.
taxi_zone_spatial_modeling_features_df = pd.DataFrame(
    taxi_zones_geo_gdf[["location_id"] + [column for column in static_spatial_columns if column != "canonical_location_id"]]
)

spatial_context_shape_df = pd.DataFrame(
    {
        "metric": [
            "output_rows", "distinct_location_ids", "distinct_canonical_location_ids",
            "alias_rows", "null_geometry_rows",
        ],
        "value": [
            len(taxi_zone_spatial_context_gdf),
            taxi_zone_spatial_context_gdf["location_id"].nunique(),
            taxi_zone_spatial_context_gdf["canonical_location_id"].nunique(),
            (taxi_zone_spatial_context_gdf["location_id"] != taxi_zone_spatial_context_gdf["canonical_location_id"]).sum(),
            taxi_zone_spatial_context_gdf["geometry"].isna().sum(),
        ],
    }
)

display(spatial_context_shape_df)

,metric,value
0,output_rows,264
1,distinct_location_ids,264
2,distinct_canonical_location_ids,262
3,alias_rows,2
4,null_geometry_rows,2


Let's validate the handoff grain and confirm that all physical and alias zones received their static spatial attributes before anything is written\.

### Validate Spatial Context Contract

This compact contract check protects the join into \`1\.3\.1\`: LocationIDs must remain unique, aliases must inherit canonical features, and only the expected nonphysical zones may lack geometry\-derived context\.

In [27]:
# ---------------------------------------------------------------------
# Validate Taxi Zone spatial context contract
# ---------------------------------------------------------------------

physical_context_mask = taxi_zone_spatial_context_gdf["geometry"].notna()
alias_context_mask = (
    taxi_zone_spatial_context_gdf["location_id"]
    != taxi_zone_spatial_context_gdf["canonical_location_id"]
)
unknown_location_ids = [264, 265]
unknown_mask = taxi_zone_spatial_context_gdf["location_id"].isin(unknown_location_ids)
graph_feature_columns = [
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

assert not taxi_zone_spatial_context_gdf["location_id"].duplicated().any(), (
    "Final spatial-context output must remain unique at the LocationID grain."
)
assert len(taxi_zone_spatial_context_gdf) == taxi_zones_working_gdf["location_id"].nunique(), (
    "Final spatial-context output must preserve the source LocationID count."
)
assert taxi_zone_spatial_context_gdf.loc[physical_context_mask, "canonical_location_id"].nunique() == 260, (
    "Physical Taxi Zones must preserve the 260 canonical LocationIDs."
)
assert taxi_zone_spatial_context_gdf.loc[physical_context_mask, graph_feature_columns].notna().all().all(), (
    "All 260 physical Taxi Zones must receive non-null graph features after propagation."
)
assert taxi_zone_spatial_context_gdf.loc[unknown_mask, graph_feature_columns].isna().all().all(), (
    "Unknown LocationIDs 264 and 265 must remain null for graph-derived features."
)
assert taxi_zone_spatial_context_gdf.loc[physical_context_mask, "network_betweenness_centrality"].between(0, 1).all(), (
    "Betweenness centrality must lie within [0, 1]."
)
assert taxi_zone_spatial_context_gdf.loc[physical_context_mask, "network_closeness_centrality"].between(0, 1).all(), (
    "Closeness centrality must lie within [0, 1]."
)
assert taxi_zone_spatial_context_gdf.loc[physical_context_mask, "network_clustering_coefficient"].between(0, 1).all(), (
    "Clustering coefficient must lie within [0, 1]."
)

alias_graph_feature_match_count = 0
for _, row in taxi_zone_spatial_context_gdf.loc[alias_context_mask, ["location_id", "canonical_location_id"]].drop_duplicates().iterrows():
    alias_values = taxi_zone_spatial_context_gdf.loc[
        taxi_zone_spatial_context_gdf["location_id"] == row["location_id"],
        graph_feature_columns,
    ].iloc[0]
    canonical_values = taxi_zone_spatial_context_gdf.loc[
        taxi_zone_spatial_context_gdf["location_id"] == row["canonical_location_id"],
        graph_feature_columns,
    ].iloc[0]
    assert alias_values.equals(canonical_values), (
        f"Alias LocationID {row['location_id']} must inherit the same graph features as canonical LocationID {row['canonical_location_id']}."
    )
    alias_graph_feature_match_count += 1

spatial_context_validation_df = pd.DataFrame(
    {
        "check": [
            "unique_location_id_key",
            "source_row_count_preserved",
            "canonical_geometry_count_preserved",
            "physical_rows_have_spatial_category",
            "physical_rows_have_cbd_distance",
            "physical_rows_have_connected_neighbor_count",
            "physical_rows_have_graph_metrics",
            "unknown_rows_keep_null_graph_features",
            "alias_rows_inherit_spatial_category",
            "alias_rows_inherit_graph_features",
            "connectivity_key_unique",
        ],
        "status": [
            True,
            True,
            True,
            taxi_zone_spatial_context_gdf.loc[physical_context_mask, "cbd_spatial_category"].notna().all(),
            taxi_zone_spatial_context_gdf.loc[physical_context_mask, "distance_to_cbd_miles"].notna().all(),
            taxi_zone_spatial_context_gdf.loc[physical_context_mask, "connected_neighbor_count"].notna().all(),
            taxi_zone_spatial_context_gdf.loc[physical_context_mask, graph_feature_columns[1:]].notna().all().all(),
            True,
            taxi_zone_spatial_context_gdf.loc[alias_context_mask, "cbd_spatial_category"].notna().all(),
            alias_graph_feature_match_count == alias_context_mask.sum(),
            not taxi_zone_connectivity_df[["location_id", "connected_location_id"]].duplicated().any(),
        ],
    }
)

graph_feature_non_null_summary_df = pd.DataFrame(
    {
        "grain": [
            "canonical_physical_taxi_zones",
            "canonical_physical_taxi_zones",
            "canonical_physical_taxi_zones",
            "canonical_physical_taxi_zones",
            "final_location_id_output",
            "final_location_id_output",
            "final_location_id_output",
            "final_location_id_output",
        ],
        "field_name": [
            "connected_neighbor_count",
            "network_betweenness_centrality",
            "network_closeness_centrality",
            "network_clustering_coefficient",
            "connected_neighbor_count",
            "network_betweenness_centrality",
            "network_closeness_centrality",
            "network_clustering_coefficient",
        ],
        "non_null_count": [
            taxi_zones_geo_gdf["connected_neighbor_count"].notna().sum(),
            taxi_zones_geo_gdf["network_betweenness_centrality"].notna().sum(),
            taxi_zones_geo_gdf["network_closeness_centrality"].notna().sum(),
            taxi_zones_geo_gdf["network_clustering_coefficient"].notna().sum(),
            taxi_zone_spatial_context_gdf["connected_neighbor_count"].notna().sum(),
            taxi_zone_spatial_context_gdf["network_betweenness_centrality"].notna().sum(),
            taxi_zone_spatial_context_gdf["network_closeness_centrality"].notna().sum(),
            taxi_zone_spatial_context_gdf["network_clustering_coefficient"].notna().sum(),
        ],
    }
)

zero_closeness_diagnostic_df = (
    taxi_zone_spatial_context_gdf.loc[
        physical_context_mask,
        [
            "location_id",
            "canonical_location_id",
            "zone",
            "borough",
            "network_closeness_centrality",
            "connected_neighbor_count",
        ],
    ]
    .query("network_closeness_centrality == 0")
    .sort_values(["borough", "zone", "location_id"])
    .reset_index(drop=True)
)

zero_closeness_summary_df = pd.DataFrame(
    {
        "metric": [
            "physical_zones_with_zero_closeness",
            "physical_zones_with_positive_closeness",
        ],
        "value": [
            len(zero_closeness_diagnostic_df),
            physical_context_mask.sum() - len(zero_closeness_diagnostic_df),
        ],
    }
)

display(spatial_context_validation_df)

,check,status
0,unique_location_id_key,True
1,source_row_count_preserved,True
2,canonical_geometry_count_preserved,True
3,physical_rows_have_spatial_category,True
4,physical_rows_have_cbd_distance,True
5,physical_rows_have_connected_neighbor_count,True
6,physical_rows_have_graph_metrics,True
7,unknown_rows_keep_null_graph_features,True
8,alias_rows_inherit_spatial_category,True
9,alias_rows_inherit_graph_features,True


Findings\. Everything passed\. The spatial\-context handoff is structurally clean: row counts are preserved, aliases inherited the right static features, Unknown stays null where it should, and the new graph fields are landing where expected\.

In [28]:
# ---------------------------------------------------------------------
# Summarize graph-feature non-null coverage
# ---------------------------------------------------------------------

display(graph_feature_non_null_summary_df)

,grain,field_name,non_null_count
0,canonical_physical_taxi_zones,connected_neighbor_count,260
1,canonical_physical_taxi_zones,network_betweenness_centrality,260
2,canonical_physical_taxi_zones,network_closeness_centrality,260
3,canonical_physical_taxi_zones,network_clustering_coefficient,260
4,final_location_id_output,connected_neighbor_count,262
5,final_location_id_output,network_betweenness_centrality,262
6,final_location_id_output,network_closeness_centrality,262
7,final_location_id_output,network_clustering_coefficient,262


Findings\. Coverage looks exactly right\. All 260 physical canonical Taxi Zones have the four graph fields populated, and the final LocationID output has 262 populated rows because only the two nonphysical Unknown records remain intentionally null\.

In [29]:
# ---------------------------------------------------------------------
# Summarize zero-closeness cases
# ---------------------------------------------------------------------

display(zero_closeness_summary_df)

,metric,value
0,physical_zones_with_zero_closeness,2
1,physical_zones_with_positive_closeness,260


Findings\. Only two physical rows have zero closeness, so this is a tiny edge case rather than a broader network problem\. Nearly the entire geography still has positive graph reach into the connected network\.

In [30]:
# ---------------------------------------------------------------------
# Review zero-closeness Taxi Zones
# ---------------------------------------------------------------------

if len(zero_closeness_diagnostic_df) > 0:
    display(zero_closeness_diagnostic_df)
else:
    display(pd.DataFrame(columns=zero_closeness_diagnostic_df.columns))

,location_id,canonical_location_id,zone,borough,network_closeness_centrality,connected_neighbor_count
0,103,103,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.0,0.0
1,105,103,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.0,0.0


Findings\. The zero\-closeness cases are both Governor’s Island / Ellis Island / Liberty Island, which is consistent with the disconnected\-component logic we already surfaced earlier\. So this reads like expected island geography, not a broken feature build\.

### Data Dictionary

is\_cbd\_zone — Flag for Taxi Zones inside the approximate congestion\-pricing zone\.
is\_cbd\_gateway\_zone — Flag for Taxi Zones that serve as transportation gateways into the CP zone\.
is\_cbd\_adjacent\_zone — Flag for Taxi Zones connected to the CP zone but not inside it or classified as gateways\.
cbd\_spatial\_category — Category label: cbd, gateway\_to\_cbd, adjacent\_to\_cbd, or non\_cbd\.
distance\_to\_cbd\_miles — Distance from each Taxi Zone centroid to the nearest CP\-zone polygon\.
distance\_to\_gateway\_miles — Distance from each Taxi Zone centroid to the nearest gateway\-zone polygon\.
connected\_neighbor\_count — Number of directly connected Taxi Zones in the final transportation\-aware connectivity network\.
network\_betweenness\_centrality — Normalized share of shortest paths that pass through the Taxi Zone in the final connectivity graph\.
network\_closeness\_centrality — Inverse average shortest\-path distance from the Taxi Zone to all other Taxi Zones in the final connectivity graph\.
network\_clustering\_coefficient — Share of realized neighbor\-to\-neighbor links around the Taxi Zone in the final connectivity graph\.

## 1\.2\.6\.5 Visual QA

These maps provide a lightweight visual check of the static handoff\. The CBD\-distance map should show a sensible outward gradient, while the connected\-neighbor map should reflect denser connectivity in central areas and fewer links around geographic edges and islands\.

### Map Distance to CP Zone

This block maps distance\_to\_cbd\_miles by Taxi Zone\. It helps confirm that the CP\-zone distance feature behaves geographically as expected\.

In [31]:
# ---------------------------------------------------------------------
# Map distance to CP zone with the portable Plotly MapLibre stack
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Map distance to CP zone
# ---------------------------------------------------------------------

visual_qa_gdf = (
    taxi_zones_geo_gdf[
        ["location_id", "zone", "borough", "geometry"]
    ]
    .drop_duplicates(subset=["location_id"])
    .merge(
        taxi_zone_spatial_modeling_features_df,
        on="location_id",
        how="left",
    )
)

visual_qa_gdf = gpd.GeoDataFrame(
    visual_qa_gdf,
    geometry="geometry",
    crs="EPSG:2263",
)

# Match 4.6.1: simplify in projected feet before returning to lat/lon.
visual_qa_gdf["geometry"] = (
    visual_qa_gdf["geometry"].simplify(
        tolerance=75,
        preserve_topology=True,
    )
)

visual_qa_gdf = (
    visual_qa_gdf
    .to_crs("EPSG:4326")
    .sort_values("location_id")
    .drop_duplicates(subset=["location_id"])
    .reset_index(drop=True)
)

visual_qa_gdf["location_id"] = (
    pd.to_numeric(
        visual_qa_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

distance_map_geojson = json.loads(
    visual_qa_gdf[
        [
            "location_id",
            "zone",
            "borough",
            "geometry",
        ]
    ].to_json()
)

valid_distance_values = (
    visual_qa_gdf["distance_to_cbd_miles"]
    .dropna()
)

distance_zmin = float(
    valid_distance_values.min()
)
distance_zmax = float(
    valid_distance_values.max()
)

distance_colorscale = [
    [0.0, BRAND_COLORS["ice"]],
    [0.5, BRAND_COLORS["seafoam"]],
    [1.0, BRAND_COLORS["dark_teal"]],
]

distance_hover_df = visual_qa_gdf[
    [
        "zone",
        "borough",
        "is_cbd_zone",
        "is_cbd_gateway_zone",
        "is_cbd_adjacent_zone",
        "connected_neighbor_count",
    ]
].copy()

fig = go.Figure(
    go.Choroplethmap(
        geojson=distance_map_geojson,
        locations=visual_qa_gdf[
            "location_id"
        ].tolist(),
        z=visual_qa_gdf[
            "distance_to_cbd_miles"
        ].tolist(),
        featureidkey="properties.location_id",
        colorscale=distance_colorscale,
        zmin=distance_zmin,
        zmax=distance_zmax,
        marker={
            "opacity": 0.75,
            "line": {
                "width": 0.6,
                "color": "white",
            },
        },
        colorbar={
            "title": "Distance to CP Zone (miles)",
            "thickness": 14,
            "len": 0.72,
        },
        customdata=distance_hover_df.to_numpy(
            dtype=object
        ),
        hovertemplate=(
            "LocationID: %{location}<br>"
            "Zone: %{customdata[0]}<br>"
            "Borough: %{customdata[1]}<br>"
            "Distance to CP Zone: %{z:.2f} mi<br>"
            "CP Zone: %{customdata[2]}<br>"
            "Gateway: %{customdata[3]}<br>"
            "Adjacent: %{customdata[4]}<br>"
            "Connected Neighbor Count: "
            "%{customdata[5]}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Distance to CP Zone",
    map={
        "style": "carto-positron",
        "center": {"lat": 40.72, "lon": -73.95},
        "zoom": 10,
    },
    width=950,
    height=700,
    margin={"l": 15, "r": 20, "t": 80, "b": 15},
)

apply_branding(fig)

fig.show()

Findings\. The map shows the expected distance gradient radiating outward from the CP zone, which confirms the distance feature is behaving sensibly\.

### Map Neighbor Count

This block maps connected\_neighbor\_count by Taxi Zone\. It visually checks whether the final transportation\-aware connectivity network produces sensible local structure before we rely on it for spillover features\.

In [32]:
# ---------------------------------------------------------------------
# Map connected neighbor count with the portable Plotly MapLibre stack
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Map connected neighbor count
# ---------------------------------------------------------------------

connected_neighbor_visual_qa_gdf = (
    taxi_zones_geo_gdf[
        ["location_id", "zone", "borough", "geometry"]
    ]
    .drop_duplicates(subset=["location_id"])
    .merge(
        taxi_zone_spatial_modeling_features_df[
            [
                "location_id",
                "connected_neighbor_count",
                "is_cbd_zone",
                "is_cbd_gateway_zone",
                "is_cbd_adjacent_zone",
            ]
        ],
        on="location_id",
        how="left",
    )
)

connected_neighbor_visual_qa_gdf = gpd.GeoDataFrame(
    connected_neighbor_visual_qa_gdf,
    geometry="geometry",
    crs="EPSG:2263",
)

# Match 4.6.1: simplify in projected feet before returning to lat/lon.
connected_neighbor_visual_qa_gdf[
    "geometry"
] = (
    connected_neighbor_visual_qa_gdf[
        "geometry"
    ].simplify(
        tolerance=75,
        preserve_topology=True,
    )
)

connected_neighbor_visual_qa_gdf = (
    connected_neighbor_visual_qa_gdf
    .to_crs("EPSG:4326")
    .sort_values("location_id")
    .drop_duplicates(subset=["location_id"])
    .reset_index(drop=True)
)

connected_neighbor_visual_qa_gdf[
    "location_id"
] = (
    pd.to_numeric(
        connected_neighbor_visual_qa_gdf[
            "location_id"
        ],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

connected_neighbor_geojson = json.loads(
    connected_neighbor_visual_qa_gdf[
        [
            "location_id",
            "zone",
            "borough",
            "geometry",
        ]
    ].to_json()
)

valid_connected_neighbor_values = (
    connected_neighbor_visual_qa_gdf[
        "connected_neighbor_count"
    ]
    .dropna()
)

connected_neighbor_zmin = float(
    valid_connected_neighbor_values.min()
)
connected_neighbor_zmax = float(
    valid_connected_neighbor_values.max()
)

connected_neighbor_colorscale = [
    [0.0, BRAND_COLORS["ice"]],
    [0.5, BRAND_COLORS["pale_peach"]],
    [1.0, BRAND_COLORS["terracotta"]],
]

connected_neighbor_hover_df = (
    connected_neighbor_visual_qa_gdf[
        [
            "zone",
            "borough",
            "is_cbd_zone",
            "is_cbd_gateway_zone",
            "is_cbd_adjacent_zone",
        ]
    ].copy()
)

fig = go.Figure(
    go.Choroplethmap(
        geojson=connected_neighbor_geojson,
        locations=connected_neighbor_visual_qa_gdf[
            "location_id"
        ].tolist(),
        z=connected_neighbor_visual_qa_gdf[
            "connected_neighbor_count"
        ].tolist(),
        featureidkey="properties.location_id",
        colorscale=connected_neighbor_colorscale,
        zmin=connected_neighbor_zmin,
        zmax=connected_neighbor_zmax,
        marker={
            "opacity": 0.75,
            "line": {
                "width": 0.6,
                "color": "white",
            },
        },
        colorbar={
            "title": "Connected Neighbor Count",
            "thickness": 14,
            "len": 0.72,
        },
        customdata=(
            connected_neighbor_hover_df
            .to_numpy(dtype=object)
        ),
        hovertemplate=(
            "LocationID: %{location}<br>"
            "Zone: %{customdata[0]}<br>"
            "Borough: %{customdata[1]}<br>"
            "Connected Neighbor Count: %{z}<br>"
            "CP Zone: %{customdata[2]}<br>"
            "Gateway: %{customdata[3]}<br>"
            "Adjacent: %{customdata[4]}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Connected Neighbor Count",
    map={
        "style": "carto-positron",
        "center": {"lat": 40.72, "lon": -73.95},
        "zoom": 10,
    },
    width=950,
    height=700,
    margin={"l": 15, "r": 20, "t": 80, "b": 15},
)

apply_branding(fig)

fig.show()

Findings\. The neighbor count map looks sensible, with higher connectivity in dense central areas and lower connectivity around edge zones and islands\.

## 1\.2\.6\.6 Write Final Outputs

The final handoff keeps geography and connectivity separate: one LocationID\-level spatial context table for direct joins, one canonical edge table for later neighborhood features, and one small manifest describing the static fields\.

In [33]:
# ---------------------------------------------------------------------
# Write final static spatial context outputs
# ---------------------------------------------------------------------

spatial_context_manifest_df = pd.DataFrame(
    {
        "field_name": [
            "is_cbd_zone",
            "is_cbd_gateway_zone",
            "is_cbd_adjacent_zone",
            "cbd_spatial_category",
            "distance_to_cbd_miles",
            "distance_to_gateway_miles",
            "connected_neighbor_count",
            "network_betweenness_centrality",
            "network_closeness_centrality",
            "network_clustering_coefficient",
        ],
        "definition": [
            "Taxi Zone is inside the approximate congestion-pricing geography.",
            "Taxi Zone is a transportation gateway into the CBD.",
            "Taxi Zone is connected to the CBD but is not inside it or a gateway.",
            "Static category: cbd, gateway_to_cbd, adjacent_to_cbd, or non_cbd.",
            "Centroid distance in miles to the nearest CBD geometry.",
            "Centroid distance in miles to the nearest gateway geometry.",
            "Count of directly connected canonical Taxi Zones in the final transportation-aware graph.",
            "Normalized share of shortest paths that pass through the Taxi Zone in the final connectivity graph.",
            "Inverse average shortest-path distance from the Taxi Zone to all other Taxi Zones in the final connectivity graph.",
            "Share of realized neighbor-to-neighbor links around the Taxi Zone in the final connectivity graph.",
        ],
        "grain": ["Taxi Zone (LocationID)"] * 10,
        "feature_family": [
            "policy_geography",
            "policy_geography",
            "policy_geography",
            "policy_geography",
            "distance",
            "distance",
            "graph_connectivity",
            "graph_connectivity",
            "graph_connectivity",
            "graph_connectivity",
        ],
        "created_in_notebook": ["1.2.6"] * 10,
    }
)

if WRITE_OUTPUTS and (
    REBUILD_SPATIAL_CONTEXT
    or not TAXI_ZONE_SPATIAL_CONTEXT_OUTPUT_PATH.exists()
    or not TAXI_ZONE_CONNECTIVITY_OUTPUT_PATH.exists()
    or not SPATIAL_CONTEXT_MANIFEST_OUTPUT_PATH.exists()
):
    spatial_context_output_gdf = taxi_zone_spatial_context_gdf.copy()
    spatial_context_output_gdf["geometry"] = spatial_context_output_gdf["geometry"].apply(
        lambda geometry: geometry.wkb if geometry is not None else None
    )
    pd.DataFrame(spatial_context_output_gdf).to_parquet(
        TAXI_ZONE_SPATIAL_CONTEXT_OUTPUT_PATH,
        index=False,
    )
    taxi_zone_connectivity_df.to_parquet(
        TAXI_ZONE_CONNECTIVITY_OUTPUT_PATH,
        index=False,
    )
    spatial_context_manifest_df.to_csv(
        SPATIAL_CONTEXT_MANIFEST_OUTPUT_PATH,
        index=False,
    )
    export_status = "written"
else:
    export_status = "reused_existing_outputs"

spatial_output_status_df = pd.DataFrame(
    {
        "artifact": [
            "taxi_zone_spatial_context",
            "taxi_zone_connectivity",
            "spatial_context_manifest",
        ],
        "path": [
            str(TAXI_ZONE_SPATIAL_CONTEXT_OUTPUT_PATH),
            str(TAXI_ZONE_CONNECTIVITY_OUTPUT_PATH),
            str(SPATIAL_CONTEXT_MANIFEST_OUTPUT_PATH),
        ],
        "status": [export_status] * 3,
    }
)

display(spatial_output_status_df)

/tmp/ipykernel_342/2630176095.py:55: UserWarning: Geometry column does not contain geometry.
  spatial_context_output_gdf["geometry"] = spatial_context_output_gdf["geometry"].apply(


,artifact,path,status
0,taxi_zone_spatial_context,pipeline_data/1.2.6.final_tables/taxi_zone_spa...,written
1,taxi_zone_connectivity,pipeline_data/1.2.6.final_tables/taxi_zone_con...,written
2,spatial_context_manifest,pipeline_data/1.2.6.final_tables/spatial_conte...,written


In [34]:
import gc

gc.collect()
print("1.2.6 spatial context build complete.")

1.2.6 spatial context build complete.


## Close

This notebook turns the harmonized Taxi Zone geography into a reusable static context layer for the rest of the project\. We cleaned up multipart and alias geometry, built a transportation\-aware connectivity network, defined the project’s policy\-geography categories, and added distance plus graph\-based connectivity features that later notebooks can reuse without rebuilding spatial logic\. The result is a stable spatial sidecar that \`1\.3\.1\` and the downstream modeling notebooks can join in directly\.

In [35]:
# Throwaway QA: confirm new 1.2.6 graph fields were written to final outputs

spatial_context_check_df = pd.read_parquet(TAXI_ZONE_SPATIAL_CONTEXT_OUTPUT_PATH)
manifest_check_df = pd.read_csv(SPATIAL_CONTEXT_MANIFEST_OUTPUT_PATH)

expected_graph_columns = [
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

print("=== COLUMN PRESENCE ===")
for column_name in expected_graph_columns:
    print(f"{column_name}: {column_name in spatial_context_check_df.columns}")

print("\n=== NON-NULL COUNTS ===")
display(
    pd.DataFrame(
        {
            "column_name": expected_graph_columns,
            "non_null_count": [
                spatial_context_check_df[column_name].notna().sum()
                if column_name in spatial_context_check_df.columns else np.nan
                for column_name in expected_graph_columns
            ],
        }
    )
)

print("\n=== UNKNOWN ROW CHECK ===")
display(
    spatial_context_check_df.loc[
        spatial_context_check_df["location_id"].isin([264, 265]),
        ["location_id"] + expected_graph_columns,
    ]
)

print("\n=== MANIFEST CHECK ===")
display(
    manifest_check_df.loc[
        manifest_check_df["field_name"].isin(expected_graph_columns)
    ]
)

=== COLUMN PRESENCE ===
connected_neighbor_count: True
network_betweenness_centrality: True
network_closeness_centrality: True
network_clustering_coefficient: True

=== NON-NULL COUNTS ===


,column_name,non_null_count
0,connected_neighbor_count,262
1,network_betweenness_centrality,262
2,network_closeness_centrality,262
3,network_clustering_coefficient,262



=== UNKNOWN ROW CHECK ===


,location_id,connected_neighbor_count,network_betweenness_centrality,network_closeness_centrality,network_clustering_coefficient
262,264,NaN,NaN,NaN,NaN
263,265,NaN,NaN,NaN,NaN



=== MANIFEST CHECK ===


,field_name,definition,grain,feature_family,created_in_notebook
6,connected_neighbor_count,Count of directly connected canonical Taxi Zon...,Taxi Zone (LocationID),graph_connectivity,1.2.6
7,network_betweenness_centrality,Normalized share of shortest paths that pass t...,Taxi Zone (LocationID),graph_connectivity,1.2.6
8,network_closeness_centrality,Inverse average shortest-path distance from th...,Taxi Zone (LocationID),graph_connectivity,1.2.6
9,network_clustering_coefficient,Share of realized neighbor-to-neighbor links a...,Taxi Zone (LocationID),graph_connectivity,1.2.6


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>